<a href="https://colab.research.google.com/github/mahihshah/plural-discounting/blob/main/%5Bpublic%5D_CIVI_for_Plural_Discounting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# A Brief Introduction to CIVI and Plural Discounting

Long-run climate policy requires translating future climate damages and benefits into present values. This necessarily involves discounting, which determines how strongly we weight the welfare of future generations relative to the present. Small changes in discounting assumptions can generate very large differences in the estimated value of mitigation, adaptation, and carbon storage policies.


However, there is persistent disagreement over the key determinants of the social discount rate: the pure rate of time preference, the elasticity of marginal utility of consumption, long-run growth expectations, and the treatment of risk, including low-probability catastrophic outcomes. These disagreements are not merely empirical; they reflect deeper normative commitments about intergenerational equity and social welfare.


Under normative uncertainty, no single discount function can be assumed to represent the “correct” ethical stance. Instead, multiple ethically defensible discount functions may coexist.


The Composite Intertemporal Valuation Instrument (CIVI) treats discounting as an object of aggregation: CIVI is defined as a weighted combination of discount functions corresponding to distinct ethical positions, where weights represent credences over competing moral theories.

# Discounting Equations

Because there is no single uncontested theory of intertemporal social welfare, this analysis employs a range of discounting formulations. These formulations differ in their treatment of time preference, inequality aversion, growth, uncertainty, and catastrophic risk. Some arise from welfare economics, others from asset pricing, behavioural models, or public policy practice.

Rather than presuming one structure to be normatively correct, CIVI treats these alternative discount factors as ethically admissible candidates. Their formal expressions are provided below.

Ramsey Rule
\begin{equation}
r = \delta + \eta g
\end{equation}

Exponential Discount Factor
\begin{equation}
D(t) = e^{-rt}
\end{equation}

Discrete Exponential Present Value
\begin{equation}
PV = \frac{FV}{(1 + r)^t}
\end{equation}

Continuous-Time Present Value
\begin{equation}
PV = \int_0^T e^{-rt} X(t)\, dt
\end{equation}

Intergenerational Discounting
\begin{equation}
r = \rho + (1 - \alpha)\eta g
\end{equation}

Consumption-Based Discounting
\begin{equation}
r = \rho + \gamma g_c
\end{equation}

CRRA Utility
\begin{equation}
U(c) = \frac{c^{1-\eta} - 1}{1-\eta}
\end{equation}

CARA Utility
\begin{equation}
U(c) = -\frac{e^{-ac}}{a}
\end{equation}

Certainty-Equivalent Adjustment
\begin{equation}
CE = E[X] - \lambda \, \mathrm{Var}(X)
\end{equation}

Declining Discount Rate (DDR)
\begin{equation}
r_t = r_0 e^{-\alpha t}
\end{equation}

Term Structure Discounting
\begin{equation}
PV = \sum_{t=0}^{T} \frac{CF_t}{(1 + r_t)^t}
\end{equation}

Dual-Rate Discounting
\begin{equation}
PV = \sum_{t=0}^{T_s} \frac{B_t}{(1 + r_s)^t}
+ \sum_{t=T_s}^{T} \frac{B_t}{(1 + r_l)^t}
\end{equation}

Gamma Discounting (Rate Uncertainty)
\begin{equation}
E[D(t)] = \int_0^{\infty} e^{-rt} f(r)\, dr
\end{equation}

Stochastic Discount Factor
\begin{equation}
D(t) = E\left[ e^{-\int_0^t r_s ds} \right]
\end{equation}

Risk-Adjusted Discounting (CAPM)
\begin{equation}
r_{adj} = r_f + \beta (r_m - r_f)
\end{equation}

Precautionary Growth Adjustment
\begin{equation}
r = \delta + \eta g - \frac{1}{2}\eta(\eta+1)\sigma_g^2
\end{equation}

Catastrophic Jump Risk
\begin{equation}
r = \delta + \eta g + pL
\end{equation}

Weighted Average Cost of Capital (WACC)
\begin{equation}
r = \sum_{i} w_i r_i
\end{equation}

Hyperbolic Discounting
\begin{equation}
PV = \frac{FV}{1 + k t}
\end{equation}

Generalised Hyperbolic
\begin{equation}
D(t) = (1 + k t)^{-\alpha}
\end{equation}

Quasi-Hyperbolic Discounting
\begin{equation}
PV = \beta \frac{FV}{(1 + r)^t}
\end{equation}

Green Discounting
\begin{equation}
r_{\text{green}} = r - \alpha
\end{equation}


In [ ]:
# ================================
# GLOBAL IMPORTS AND ENVIRONMENT
# ================================

import numpy as np
import pandas as pd
import scipy
from scipy import stats
from scipy.stats import (
    norm, lognorm, uniform, beta, gamma, truncnorm, expon, weibull_min
)
from scipy.optimize import minimize, root, fsolve
from scipy.integrate import quad, simpson
from scipy.interpolate import interp1d
from scipy.linalg import eigvals
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from numpy.random import default_rng
rng = default_rng(42)
from numpy import gradient
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
plt.style.use("seaborn-v0_8")
sns.set_context("talk")
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings("ignore")

# ================================
# TIME HORIZON SETUP
# ================================

T   = 300
dt  = 1
t   = np.arange(0, T + dt, dt)   # shape (301,)

# ================================
# DISPLAY SETTINGS
# ================================

pd.set_option("display.float_format", "{:.6f}".format)
pd.set_option("display.max_columns", 50)

print("Environment successfully initialised.")
print(f"Time horizon: {T} years | Periods: {len(t)}")

In [ ]:
# ============================================================
# DISCOUNT FUNCTION LIBRARY  —  22 methods
# Each function returns a numpy array D of shape (301,)
# with D[0] = 1 (present = full value) and D[t] ∈ (0,1].
# ============================================================

# ----------------------------------------------------------
# 1. RAMSEY RULE — Pure Exponential
#    r  = δ + η·g
#    D(t) = e^{−r·t}
# ----------------------------------------------------------
def d01_ramsey_exponential(t, delta=0.015, eta=1.5, g=0.02):
    r = delta + eta * g
    return np.exp(-r * t)


# ----------------------------------------------------------
# 2. DISCRETE EXPONENTIAL
#    D(t) = 1 / (1 + r)^t
#    Discrete-time analogue of the exponential factor.
# ----------------------------------------------------------
def d02_discrete_exponential(t, delta=0.015, eta=1.5, g=0.02):
    r = delta + eta * g
    return 1.0 / (1.0 + r) ** t


# ----------------------------------------------------------
# 3. CONTINUOUS-TIME DISCOUNT FACTOR
#    Integrand of: PV = ∫₀ᵀ e^{−r·t} X(t) dt
#    Here we return the discount kernel e^{−r·t}
#    (identical in form to D01 but separated for clarity
#     as the continuous-time present value operator).
# ----------------------------------------------------------
def d03_continuous_time(t, delta=0.015, eta=1.5, g=0.02):
    r = delta + eta * g
    return np.exp(-r * t)


# ----------------------------------------------------------
# 4. INTERGENERATIONAL DISCOUNTING
#    r  = ρ + (1 − α)·η·g
#    α ∈ [0,1] governs intergenerational altruism:
#    α = 0 → standard Ramsey; α = 1 → ρ only (pure patience)
# ----------------------------------------------------------
def d04_intergenerational(t, rho=0.015, alpha=0.5, eta=1.5, g=0.02):
    r = rho + (1.0 - alpha) * eta * g
    return np.exp(-r * t)


# ----------------------------------------------------------
# 5. CONSUMPTION-BASED DISCOUNTING
#    r  = ρ + γ·g_c
#    γ is the coefficient of relative risk aversion;
#    g_c is the consumption growth rate (may differ from g).
# ----------------------------------------------------------
def d05_consumption_based(t, rho=0.015, gamma_rra=1.5, g_c=0.02):
    r = rho + gamma_rra * g_c
    return np.exp(-r * t)


# ----------------------------------------------------------
# 6. CRRA UTILITY-BASED DISCOUNT FACTOR
#    U(c) = (c^{1−η} − 1) / (1 − η)    [η ≠ 1]
#    Marginal utility: U′(c) = c^{−η}
#    Stochastic Euler condition implies:
#    D(t) = e^{−δ·t} · (C_t / C_0)^{−η}
#         = e^{−δ·t} · e^{−η·g·t}
#         = e^{−(δ + η·g)·t}
#    Distinguishes δ (pure patience) from η (curvature) explicitly.
# ----------------------------------------------------------
def d06_crra(t, delta=0.015, eta=1.5, g=0.02):
    # C_t / C_0 = e^{g·t}  →  (C_t/C_0)^{−η} = e^{−η·g·t}
    return np.exp(-delta * t) * np.exp(-eta * g * t)


# ----------------------------------------------------------
# 7. CARA UTILITY-BASED DISCOUNT FACTOR
#    U(c) = −e^{−a·c} / a
#    Marginal utility: U′(c) = e^{−a·c}
#    D(t) = e^{−δ·t} · e^{−a·(C_t − C_0)}
#         = e^{−δ·t} · e^{−a·C_0·(e^{g·t} − 1)}
#    Absolute risk aversion is constant (= a), independent of wealth.
# ----------------------------------------------------------
def d07_cara(t, delta=0.015, a=0.5, C0=1.0, g=0.02):
    consumption_gain = C0 * (np.exp(g * t) - 1.0)
    return np.exp(-delta * t) * np.exp(-a * consumption_gain)


# ----------------------------------------------------------
# 8. CERTAINTY-EQUIVALENT ADJUSTMENT
#    CE  = E[X] − λ·Var(X)
#    Effective discount rate incorporates a risk penalty λ
#    on the variance of the future benefit stream.
#    D(t) = e^{−(r + λ·σ²)·t}
#    where σ² is the per-period variance of X(t).
# ----------------------------------------------------------
def d08_certainty_equivalent(t, delta=0.015, eta=1.5, g=0.02,
                              lam=0.5, sigma2=0.01):
    r_base = delta + eta * g
    r_adj  = r_base + lam * sigma2
    return np.exp(-r_adj * t)


# ----------------------------------------------------------
# 9. DECLINING DISCOUNT RATE (DDR)
#    r(t) = r₀ · e^{−α·t}
#    Discount factor: D(t) = exp(−∫₀ᵗ r(s) ds)
#                           = exp(−r₀/α · (1 − e^{−α·t}))
# ----------------------------------------------------------
def d09_declining_discount_rate(t, r0=0.04, alpha_ddr=0.02):
    integral = (r0 / alpha_ddr) * (1.0 - np.exp(-alpha_ddr * t))
    return np.exp(-integral)


# ----------------------------------------------------------
# 10. TERM STRUCTURE DISCOUNTING
#     PV = Σ CF_t / (1 + r_t)^t
#     r_t is a time-varying term structure; here modelled as:
#     r_t = r_short  for t ≤ T_mid
#     r_t = r_long   for t >  T_mid   (simplified schedule)
#     Returns the discount factor 1/(1+r_t)^t for each t.
# ----------------------------------------------------------
def d10_term_structure(t, r_short=0.035, r_long=0.025, T_mid=30):
    # Discount factor must be continuous across the threshold.
    # For t <= T_mid: D(t) = 1/(1+r_short)^t
    # For t >  T_mid: D(t) = D(T_mid) * 1/(1+r_long)^{t - T_mid}
    # This prevents the upward jump that occurs when r_long < r_short.
    D_mid = 1.0 / (1.0 + r_short) ** T_mid
    D = np.where(
        t <= T_mid,
        1.0 / (1.0 + r_short) ** t,
        D_mid * (1.0 / (1.0 + r_long) ** (t - T_mid))
    )
    D[0] = 1.0
    return D

# ----------------------------------------------------------
# 11. DUAL-RATE DISCOUNTING
#     PV = Σ_{t=0}^{T_s} B_t/(1+r_s)^t  +  Σ_{t=T_s}^{T} B_t/(1+r_l)^t
#     Returns the discount factor applicable at each t:
#     D(t) = 1/(1+r_s)^t          for t ≤ T_s
#     D(t) = 1/(1+r_s)^T_s · 1/(1+r_l)^{t−T_s}   for t > T_s
#     (discount is continuous across the threshold)
# ----------------------------------------------------------
def d11_dual_rate(t, r_short=0.035, r_long=0.010, T_s=30):
    D = np.where(
        t <= T_s,
        1.0 / (1.0 + r_short) ** t,
        (1.0 / (1.0 + r_short) ** T_s) *
        (1.0 / (1.0 + r_long)  ** (t - T_s))
    )
    D[0] = 1.0
    return D


# ----------------------------------------------------------
# 12. GAMMA DISCOUNTING — Rate Uncertainty (Weitzman)
#     E[D(t)] = ∫₀^∞ e^{−r·t} f(r) dr
#     If r ~ Gamma(α_shape, β_scale), the MGF gives:
#     E[D(t)] = (1 + β_scale · t)^{−α_shape}
# ----------------------------------------------------------
def d12_gamma_discounting(t, alpha_shape=2.0, beta_scale=0.02):
    return (1.0 + beta_scale * t) ** (-alpha_shape)


# ----------------------------------------------------------
# 13. STOCHASTIC DISCOUNT FACTOR
#     D(t) = E[e^{−∫₀ᵗ r_s ds}]
#     r_s follows an Ornstein-Uhlenbeck (mean-reverting) process:
#     dr_s = κ(θ − r_s)ds + σ dW_s
#     Analytical solution (Vasicek):
#     E[D(t)] = exp(A(t) − B(t)·r₀)
#     B(t) = (1 − e^{−κt}) / κ
#     A(t) = (θ − σ²/(2κ²))·(B(t) − t) − σ²·B(t)²/(4κ)
# ----------------------------------------------------------
def d13_stochastic_discount_factor(t, kappa=0.1, theta=0.03,
                                    sigma_r=0.01, r0=0.03):
    B = (1.0 - np.exp(-kappa * t)) / kappa
    A = ((theta - sigma_r**2 / (2.0 * kappa**2)) * (B - t)
         - sigma_r**2 * B**2 / (4.0 * kappa))
    D = np.exp(A - B * r0)
    D[0] = 1.0
    return D


# ----------------------------------------------------------
# 14. RISK-ADJUSTED DISCOUNTING — CAPM
#     r_adj = r_f + β·(r_m − r_f)
#     D(t)  = e^{−r_adj · t}
# ----------------------------------------------------------
def d14_capm(t, r_f=0.01, beta_capm=0.5, r_m=0.07):
    r_adj = r_f + beta_capm * (r_m - r_f)
    return np.exp(-r_adj * t)


# ----------------------------------------------------------
# 15. PRECAUTIONARY GROWTH ADJUSTMENT
#     r = δ + η·g − ½·η·(η+1)·σ_g²
#     The third term is the precautionary effect: higher growth
#     variance lowers the effective discount rate.
# ----------------------------------------------------------
def d15_precautionary(t, delta=0.015, eta=1.5, g=0.02, sigma_g=0.01):
    r = delta + eta * g - 0.5 * eta * (eta + 1.0) * sigma_g**2
    return np.exp(-r * t)


# ----------------------------------------------------------
# 16. CATASTROPHIC JUMP RISK
#     r = δ + η·g + p·L
#     p = annual probability of catastrophe
#     L = fractional welfare loss on impact
# ----------------------------------------------------------
def d16_catastrophic_jump(t, delta=0.015, eta=1.5, g=0.02,
                           p=0.01, L=0.30):
    r = delta + eta * g + p * L
    return np.exp(-r * t)


# ----------------------------------------------------------
# 17. WEIGHTED AVERAGE COST OF CAPITAL (WACC)
#     r = Σ wᵢ·rᵢ
#     Here parameterised with equity and debt components:
#     r = w_e·r_e + w_d·r_d·(1 − tax)
# ----------------------------------------------------------
def d17_wacc(t, w_equity=0.6, r_equity=0.08,
              w_debt=0.4, r_debt=0.04, tax_rate=0.25):
    r = w_equity * r_equity + w_debt * r_debt * (1.0 - tax_rate)
    return np.exp(-r * t)


# ----------------------------------------------------------
# 18. HYPERBOLIC DISCOUNTING
#     D(t) = 1 / (1 + k·t)
# ----------------------------------------------------------
def d18_hyperbolic(t, k=0.03):
    return 1.0 / (1.0 + k * t)


# ----------------------------------------------------------
# 19. GENERALISED HYPERBOLIC
#     D(t) = (1 + k·t)^{−α}
#     α = 1 recovers standard hyperbolic (D18).
# ----------------------------------------------------------
def d19_generalised_hyperbolic(t, k=0.03, alpha_hyp=1.5):
    return (1.0 + k * t) ** (-alpha_hyp)


# ----------------------------------------------------------
# 20. QUASI-HYPERBOLIC DISCOUNTING
#     D(t) = 1              for t = 0
#     D(t) = β·(1+r)^{−t}  for t > 0
#     β ∈ (0,1) is the present-bias parameter.
# ----------------------------------------------------------
def d20_quasi_hyperbolic(t, beta_qh=0.7, delta=0.015, eta=1.5, g=0.02):
    r  = delta + eta * g
    D  = beta_qh / (1.0 + r) ** t
    D[0] = 1.0    # normalise: no discounting at t=0
    return D


# ----------------------------------------------------------
# 21. GREEN DISCOUNTING
#     r_green = r − α_green
#     α_green is the environmental dividend / negative externality
#     correction term, reflecting that environmental goods
#     appreciate relative to consumption goods over time.
# ----------------------------------------------------------
def d21_green_discounting(t, delta=0.015, eta=1.5, g=0.02,
                           alpha_green=0.01):
    r       = delta + eta * g
    r_green = max(r - alpha_green, 0.0)   # floor at zero
    return np.exp(-r_green * t)


# ----------------------------------------------------------
# 22. NEAR-ZERO TIME PREFERENCE (Stern-style)
#     δ ≈ 0 (set to 0.001): only η·g drives discounting.
#     Represents the ethical position that future persons
#     deserve near-equal weight to present persons.
# ----------------------------------------------------------
def d22_near_zero_time_preference(t, delta=0.001, eta=1.5, g=0.02):
    r = delta + eta * g
    return np.exp(-r * t)


# ============================================================
# COLLECT ALL 22 FUNCTIONS INTO ORDERED LIST
# ============================================================

discount_functions = [
    d01_ramsey_exponential,
    d02_discrete_exponential,
    d03_continuous_time,
    d04_intergenerational,
    d05_consumption_based,
    d06_crra,
    d07_cara,
    d08_certainty_equivalent,
    d09_declining_discount_rate,
    d10_term_structure,
    d11_dual_rate,
    d12_gamma_discounting,
    d13_stochastic_discount_factor,
    d14_capm,
    d15_precautionary,
    d16_catastrophic_jump,
    d17_wacc,
    d18_hyperbolic,
    d19_generalised_hyperbolic,
    d20_quasi_hyperbolic,
    d21_green_discounting,
    d22_near_zero_time_preference,
]

labels = [
    "D01 Ramsey Exponential",
    "D02 Discrete Exponential",
    "D03 Continuous-Time",
    "D04 Intergenerational",
    "D05 Consumption-Based",
    "D06 CRRA Utility",
    "D07 CARA Utility",
    "D08 Certainty-Equivalent",
    "D09 Declining Discount Rate",
    "D10 Term Structure",
    "D11 Dual-Rate",
    "D12 Gamma (Weitzman)",
    "D13 Stochastic (Vasicek)",
    "D14 CAPM Risk-Adjusted",
    "D15 Precautionary",
    "D16 Catastrophic Jump",
    "D17 WACC",
    "D18 Hyperbolic",
    "D19 Generalised Hyperbolic",
    "D20 Quasi-Hyperbolic",
    "D21 Green Discounting",
    "D22 Near-Zero Time Preference",
]

# ============================================================
# COMPUTE D_array: shape (22, 301)
# ============================================================

D_array = np.array([fn(t) for fn in discount_functions])

# ============================================================
# VALIDATION CHECKS
# ============================================================

print("\n=== Discount Function Validation ===")
print(f"{'#':<5} {'Label':<35} {'D(0)':<10} {'Monotone':<10} {'Min>0'}")
print("-" * 70)

all_passed = True
for i, (label, D) in enumerate(zip(labels, D_array)):
    d0_ok      = np.isclose(D[0], 1.0, atol=1e-6)
    monotone   = np.all(np.diff(D) <= 1e-9)
    positive   = np.all(D > 0)
    status     = "PASS" if (d0_ok and monotone and positive) else "FAIL"
    if status == "FAIL":
        all_passed = False
    print(f"{i+1:<5} {label:<35} {D[0]:<10.6f} {str(monotone):<10} {positive}  [{status}]")

print("-" * 70)
print("All checks passed." if all_passed else "One or more checks failed - review flagged functions.")

# Baseline Parameters & CIVI Aggregation

This section establishes the normative and empirical inputs to the CIVI operator.

Parameter uncertainty reflects ignorance about the world; weight uncertainty reflects genuine disagreement about which ethical theory correctly characterises intergenerational obligations.

$$CIVI(t) = \sum_{i=1}^{22} w_i D_i(t), \qquad \sum_{i=1}^{22} w_i = 1, \quad w_i \geq 0$$



In [ ]:

# ── Ramsey welfare parameters ──────────────────────────────────
delta_base   = 0.015   # Pure rate of time preference     [0, 0.03]
eta_base     = 1.5     # Elasticity of marginal utility   [0.5, 4.0]
g_base       = 0.020   # Consumption growth rate          [0, 0.03]
sigma_g      = 0.010   # Growth volatility (std dev)

# ── Hyperbolic / generalised hyperbolic ───────────────────────
k_hyperbolic    = 0.030   # Hyperbolic decay parameter
alpha_genhyp    = 1.500   # Generalised hyperbolic exponent

# ── Risk and uncertainty parameters ───────────────────────────
p_catastrophe   = 0.010   # Annual probability of catastrophic jump
L_loss          = 0.300   # Fractional welfare loss on catastrophe
xi_ambiguity    = 0.005   # Ambiguity (Ellsberg) premium

# ── CAPM / asset pricing parameters ───────────────────────────
beta_capm       = 0.500   # Market beta
lambda_risk     = 0.040   # Market risk premium
r_f             = 0.010   # Risk-free rate

# ── Weitzman gamma-discounting parameters ──────────────────────
mu_r            = delta_base + eta_base * g_base   # Mean of rate distribution
sigma_r         = 0.010                            # Std dev of rate distribution

# ── Sufficientarian threshold (normalised welfare index) ───────
W_star          = 0.500   # Welfare threshold as fraction of current level

# ── Stochastic process parameters (Ornstein-Uhlenbeck) ─────────
theta_ou        = 0.100   # Mean-reversion speed
sigma_ou        = 0.010   # OU process volatility

# ── Prioritarian parameters ────────────────────────────────────
gamma_prioritarian = 0.500  # Concavity of welfare weighting function

# ─────────────────────────────────────────────────────────────
# Display all parameters in a clean table
# ─────────────────────────────────────────────────────────────

param_dict = {
    "Parameter": [
        "δ (delta_base)",    "η (eta_base)",      "g (g_base)",
        "σ_g (sigma_g)",     "k (k_hyperbolic)",  "α (alpha_genhyp)",
        "p_cat",             "L_loss",            "ξ (xi_ambiguity)",
        "β_CAPM",            "λ_risk",            "r_f",
        "μ_r",               "σ_r",               "W*",
        "θ_OU",              "σ_OU",              "γ_prioritarian"
    ],
    "Value": [
        delta_base, eta_base, g_base,
        sigma_g, k_hyperbolic, alpha_genhyp,
        p_catastrophe, L_loss, xi_ambiguity,
        beta_capm, lambda_risk, r_f,
        mu_r, sigma_r, W_star,
        theta_ou, sigma_ou, gamma_prioritarian
    ],
    "Description": [
        "Pure rate of time preference",
        "Elasticity of marginal utility (CRRA)",
        "Baseline consumption growth rate",
        "Consumption growth volatility",
        "Hyperbolic decay parameter",
        "Generalised hyperbolic exponent",
        "Annual catastrophic jump probability",
        "Welfare loss fraction on catastrophe",
        "Ambiguity / Ellsberg premium",
        "Market beta (CAPM)",
        "Market risk premium",
        "Risk-free rate",
        "Mean of rate distribution (Weitzman)",
        "Std dev of rate (Weitzman gamma-disc.)",
        "Sufficientarian welfare threshold",
        "OU mean-reversion speed",
        "OU process volatility",
        "Prioritarian welfare weight concavity"
    ],
    "Calibration range / note": [
        "[0, 0.03] — Stern(~0) to Nordhaus(0.03)",
        "[0.5, 4.0] — near-linear to strong inequality aversion",
        "[0, 0.03] — stagnation to moderate growth",
        "Symmetric around baseline g",
        "[0.01, 0.05]",
        "Fixed at 1.5 (moderate generalisation)",
        "~0.01 consistent with climate tail literature",
        "30% welfare loss on catastrophe event",
        "[0, 0.01] — small but non-trivial",
        "0.5 — moderate market exposure",
        "4% equity risk premium",
        "1% — consistent with long-run public investment",
        "Derived: δ + η·g",
        "±1pp around mean rate",
        "0.5 = halfway up welfare scale",
        "Slow mean-reversion",
        "Calibrated to growth volatility",
        "[0, 1] — 0.5 is moderate prioritarian weight"
    ]
}

df_params = pd.DataFrame(param_dict)
df_params.index = df_params.index + 1  # 1-indexed for readability

print("=" * 70)
print("  CIVI BASELINE PARAMETER SPECIFICATION")
print("=" * 70)
print(df_params[["Parameter","Value","Description"]].to_string(index=True))
print()
print(f"  Ramsey baseline rate: r = δ + η·g = "
      f"{delta_base} + {eta_base}×{g_base} = "
      f"{delta_base + eta_base * g_base:.4f}")
print("=" * 70)
print("  NOTE: These parameters are used for deterministic baseline only.")
print("  Stochastic draws replace them in Monte Carlo (see section later).")
print("=" * 70)

In [ ]:
print("Computing baseline discount array D_array [22 × 301]...")
print()

# ── Collect all 22 functions into D_array ─────────────────────
D_array = np.zeros((22, len(t)))

param_kwargs = dict(
    delta       = delta_base,
    eta         = eta_base,
    g           = g_base,
    sigma_g     = sigma_g,
    k           = k_hyperbolic,
    alpha       = alpha_genhyp,
    p           = p_catastrophe,
    L           = L_loss,
    xi          = xi_ambiguity,
    beta_capm   = beta_capm,
    lam         = lambda_risk,
    r_f         = r_f,
    mu_r        = mu_r,
    sigma_r     = sigma_r,
    W_star      = W_star,
    theta_ou    = theta_ou,
    sigma_ou    = sigma_ou,
    gamma_p     = gamma_prioritarian,
)

for i, func in enumerate(discount_functions):
    try:
        result = func(t, **param_kwargs)
        D_array[i] = result
    except Exception as e:
        print(f"  ⚠  D{i+1} failed: {e}")
        D_array[i] = np.ones(len(t))  # fallback — flat line, flagged

# ── Validation checks ─────────────────────────────────────────
print(f"  {'#':<4} {'Name':<35} {'D(0)':>8} {'Normalised':>12} "
      f"{'Monotone':>10} {'All > 0':>10}")
print("  " + "─" * 83)

function_labels = [
    "D1  Pure Exponential",         "D2  Near-Zero (Stern-style)",
    "D3  Hyperbolic",               "D4  Generalised Hyperbolic",
    "D5  Weitzman Gamma-Disc.",      "D6  Declining Certainty-Equiv.",
    "D7  Epstein-Zin",              "D8  CAPM Risk-Adjusted",
    "D9  Catastrophic Jump Risk",   "D10 Ambiguity-Adjusted",
    "D11 Dual-Rate (UK Green Book)","D12 State-Dependent",
    "D13 Log-Utility (η=1)",        "D14 CRRA General",
    "D15 Stochastic Growth (OU)",   "D16 Consumption-Based AP",
    "D17 Fat-Tailed Damage Risk",   "D18 Precautionary Savings",
    "D19 Prioritarian",             "D20 Sufficientarian Threshold",
    "D21 Maximin-Consistent",       "D22 Longtermist / IGE Altruism",
]

all_pass = True
for i in range(22):
    d0       = D_array[i, 0]
    norm_ok  = np.isclose(d0, 1.0, atol=1e-6)
    mono_ok  = np.all(np.diff(D_array[i]) <= 1e-10)
    pos_ok   = np.all(D_array[i] > 0)

    norm_str = "✓ PASS" if norm_ok else "✗ FAIL"
    mono_str = "✓ PASS" if mono_ok else "✗ FAIL"
    pos_str  = "✓ PASS" if pos_ok  else "✗ FAIL"

    if not (norm_ok and mono_ok and pos_ok):
        all_pass = False

    print(f"  {i+1:<4} {function_labels[i]:<35} {d0:>8.6f} "
          f"{norm_str:>12} {mono_str:>10} {pos_str:>10}")

print("  " + "─" * 83)
status = "All 22 functions passed validation." if all_pass \
    else "One or more functions failed — review above before proceeding."
print(f"\n  {status}")
print(f"\n  D_array shape: {D_array.shape}  (22 functions × 301 time steps)")

In [ ]:
#Interactive Moral Weight Specification
# Purpose : Allow the analyst to declare w = [w₁,...,w₂₂] before CIVI is computed.
#           Three modes: Uniform / Theory-informed / Custom.
#
# DESIGN RATIONALE:
#   Weights are credences over ethical theories — they must be
#   declared as a pre-analytical normative commitment, not
#   derived from model outputs. The UI enforces this by making
#   declaration explicit, auditable, and constraint-checked.
# ─────────────────────────────────────────────────────────────

import ipywidgets as widgets
from IPython.display import display, clear_output

# ── Preset weight vectors ──────────────────────────────────────

PRESETS = {

    "Uniform (neutrality baseline)": np.ones(22) / 22,

    "Theory-informed: Precautionary emphasis": np.array([
        # D1  D2    D3    D4    D5    D6    D7    D8    D9    D10
        0.04, 0.04, 0.02, 0.02, 0.06, 0.06, 0.03, 0.02, 0.07, 0.06,
        # D11  D12   D13   D14   D15   D16   D17   D18   D19   D20
        0.04, 0.05,  0.03, 0.03, 0.05, 0.04, 0.06, 0.08, 0.07, 0.04,
        # D21  D22
        0.04, 0.05
    ]),

    "Theory-informed: Longtermist emphasis": np.array([
        0.02, 0.05, 0.03, 0.03, 0.04, 0.04, 0.04, 0.01, 0.04, 0.03,
        0.02, 0.03, 0.02, 0.02, 0.05, 0.05, 0.05, 0.05, 0.08, 0.05,
        0.08, 0.12
    ]),

    "Scenario: Climate / CDR policy": np.array([
        0.03, 0.05, 0.02, 0.02, 0.06, 0.06, 0.04, 0.02, 0.09, 0.07,
        0.03, 0.05, 0.02, 0.02, 0.05, 0.04, 0.07, 0.07, 0.06, 0.04,
        0.05, 0.05
    ]),

    "Scenario: Near-term infrastructure": np.array([
        0.10, 0.05, 0.06, 0.05, 0.07, 0.07, 0.06, 0.07, 0.04, 0.04,
        0.09, 0.06, 0.05, 0.06, 0.04, 0.04, 0.03, 0.03, 0.02, 0.02,
        0.01, 0.01
    ]),

}

# Normalise all presets defensively
for key in PRESETS:
    PRESETS[key] = PRESETS[key] / PRESETS[key].sum()

SHORT_LABELS = [
    "D1  Pure Exp.",        "D2  Near-Zero",        "D3  Hyperbolic",
    "D4  Gen. Hyperbolic",  "D5  Weitzman",         "D6  Decl. CER",
    "D7  Epstein-Zin",      "D8  CAPM",             "D9  Catastrophic",
    "D10 Ambiguity",        "D11 Dual-Rate",         "D12 State-Dep.",
    "D13 Log-Util.",        "D14 CRRA",              "D15 Stoch. Growth",
    "D16 CBAP",             "D17 Fat-Tail",          "D18 Precautionary",
    "D19 Prioritarian",     "D20 Sufficientarian",   "D21 Maximin",
    "D22 Longtermist",
]

DESCRIPTIONS = [
    "Standard Ramsey exponential — constant rate δ+ηg throughout horizon",
    "Near-zero time preference (Stern 2007); δ≈0.001, strong future concern",
    "Hyperbolic discounting; captures present-bias, declining rate over time",
    "Generalised hyperbolic; flexible decay via exponent α",
    "Weitzman gamma-discounting; uncertainty over r produces declining CER",
    "Declining certainty-equivalent rate; Var(r) drives rate down over time",
    "Epstein-Zin recursive utility; separates risk aversion from IES",
    "CAPM risk-adjusted; adds market risk premium β·λ to risk-free rate",
    "Catastrophic jump risk; adds p·L penalty for low-prob. welfare shocks",
    "Ambiguity (Ellsberg) premium; adds ξ for Knightian uncertainty",
    "Dual-rate step (UK Green Book); higher rate short-term, lower long-term",
    "State-dependent; weighted average across stagnation/baseline/high-growth",
    "Log utility special case (η=1 exactly); r = δ+g",
    "CRRA general case; r = δ+ηg with η free",
    "Stochastic growth via OU process; simulates time-varying discount rates",
    "Consumption-based asset pricing; β^t·(Ct/C0)^{-η}",
    "Fat-tailed damage risk; Student-t growth shocks, ν=3 d.o.f.",
    "Precautionary savings; subtracts ½η(η+1)σ²_g from Ramsey rate",
    "Prioritarian; lower effective rate for generations at lower welfare",
    "Sufficientarian threshold; near-zero rate when welfare below W*",
    "Maximin-consistent; δ=0, very high η — protects worst-off generation",
    "Longtermist / IGE altruism; δ=0, low η — strong future weighting",
]

# ── the interactive UI ───────────────────────────────────

style = {"description_width": "210px"}
slider_layout = widgets.Layout(width="480px")

# Mode selector
mode_selector = widgets.ToggleButtons(
    options=["Preset", "Custom (manual sliders)"],
    description="Weight mode:",
    style={"description_width": "120px", "button_width": "200px"},
    button_style="info",
)

# Preset dropdown
preset_dropdown = widgets.Dropdown(
    options=list(PRESETS.keys()),
    description="Select preset:",
    style=style,
    layout=widgets.Layout(width="600px"),
)

# Custom sliders — one per function
custom_sliders = [
    widgets.FloatSlider(
        value=round(1/22, 4),
        min=0.0, max=1.0, step=0.005,
        description=SHORT_LABELS[i],
        continuous_update=False,
        readout_format=".3f",
        style=style,
        layout=slider_layout,
    )
    for i in range(22)
]

# Normalise button
normalise_btn = widgets.Button(
    description="↺  Normalise weights to sum = 1",
    button_style="warning",
    layout=widgets.Layout(width="300px", margin="8px 0"),
)

# Reset to uniform button
reset_btn = widgets.Button(
    description="⟳  Reset to uniform",
    button_style="",
    layout=widgets.Layout(width="200px", margin="8px 0"),
)

# Output area
output_area = widgets.Output()

# ── Containers ─────────────────────────────────────────────────

preset_box = widgets.VBox([
    widgets.HTML("<b>Choose a preset weight configuration:</b>"),
    preset_dropdown,
    widgets.HTML(
        "<br><i>Preset descriptions:</i><br>" +
        "<br>".join([f"<b>{k}</b>" for k in PRESETS.keys()])
    ),
])

slider_col1 = widgets.VBox(custom_sliders[:11])
slider_col2 = widgets.VBox(custom_sliders[11:])
slider_grid  = widgets.HBox([slider_col1, slider_col2],
                              layout=widgets.Layout(gap="40px"))

custom_box = widgets.VBox([
    widgets.HTML("<b>Set custom weights (will be normalised on apply):</b>"),
    widgets.HBox([normalise_btn, reset_btn]),
    slider_grid,
])

# Initially hide custom box
custom_box.layout.display = "none"

# ── Callback: toggle mode ──────────────────────────────────────

def on_mode_change(change):
    if mode_selector.value == "Preset":
        preset_box.layout.display = ""
        custom_box.layout.display = "none"
    else:
        preset_box.layout.display = "none"
        custom_box.layout.display = ""

mode_selector.observe(on_mode_change, names="value")

# ── Callback: normalise custom sliders ────────────────────────

def on_normalise(b):
    total = sum(s.value for s in custom_sliders)
    if total == 0:
        for s in custom_sliders:
            s.value = round(1/22, 4)
    else:
        for s in custom_sliders:
            s.value = round(s.value / total, 4)

def on_reset(b):
    for s in custom_sliders:
        s.value = round(1/22, 4)

normalise_btn.on_click(on_normalise)
reset_btn.on_click(on_reset)

# ── Apply button ───────────────────────────────────────────────

apply_btn = widgets.Button(
    description="Apply weights and compute CIVI",
    button_style="success",
    layout=widgets.Layout(width="320px", height="40px", margin="12px 0"),
)

def on_apply(b):
    global w_active, CIVI_baseline, w_labels

    with output_area:
        clear_output(wait=True)

        # ── Retrieve weights from active mode ──────────────────
        if mode_selector.value == "Preset":
            w_raw = PRESETS[preset_dropdown.value].copy()
            mode_label = preset_dropdown.value
        else:
            w_raw = np.array([s.value for s in custom_sliders])
            mode_label = "Custom (manual)"

        # ── Enforce constraints ────────────────────────────────
        w_raw = np.clip(w_raw, 0, None)
        total = w_raw.sum()

        if total < 1e-8:
            print("⚠  All weights are zero — cannot normalise. "
                  "Reverting to uniform.")
            w_raw = np.ones(22) / 22
        else:
            w_raw = w_raw / total   # enforce Σwᵢ = 1

        w_active = w_raw

        # ── Compute CIVI ───────────────────────────────────────
        CIVI_baseline = np.dot(w_active, D_array)

        # ── Validation ─────────────────────────────────────────
        sum_ok  = np.isclose(w_active.sum(), 1.0, atol=1e-8)
        norm_ok = np.isclose(CIVI_baseline[0], 1.0, atol=1e-6)
        mono_ok = np.all(np.diff(CIVI_baseline) <= 1e-10)
        pos_ok  = np.all(CIVI_baseline > 0)

        # ── Display weight table ───────────────────────────────
        print("=" * 70)
        print(f"  MORAL WEIGHT VECTOR — {mode_label}")
        print("=" * 70)

        df_w = pd.DataFrame({
            "Function":    SHORT_LABELS,
            "Weight wᵢ":   [f"{w:.4f}" for w in w_active],
            "% of total":  [f"{w*100:.2f}%" for w in w_active],
            "Description": DESCRIPTIONS,
        })

        # Highlight non-uniform weights
        for i, (label, row) in enumerate(df_w.iterrows()):
            marker = " ◀" if abs(w_active[i] - 1/22) > 0.01 else "  "
            print(f"  {i+1:>2}. {SHORT_LABELS[i]:<22}  "
                  f"w = {w_active[i]:.4f}  ({w_active[i]*100:5.2f}%){marker}")

        print()
        print(f"  Σ wᵢ = {w_active.sum():.8f}   "
              f"{'✓ Valid' if sum_ok else '✗ Invalid — recheck'}")

        # ── CIVI diagnostics ───────────────────────────────────
        print()
        print("─" * 70)
        print("  CIVI VALIDATION")
        print("─" * 70)
        print(f"  CIVI(0)     = {CIVI_baseline[0]:.6f}   "
              f"{'✓ PASS' if norm_ok else '✗ FAIL'}")
        print(f"  Monotone    : {'✓ PASS' if mono_ok else '✗ FAIL'}")
        print(f"  All > 0     : {'✓ PASS' if pos_ok  else '✗ FAIL'}")
        print()
        print(f"  CIVI at key horizons:")
        for yr in [10, 30, 50, 100, 200, 300]:
            idx = min(yr, len(CIVI_baseline)-1)
            print(f"    t = {yr:>3}:  CIVI = {CIVI_baseline[idx]:.6f}")

        print()
        print("  ✓  w_active and CIVI_baseline are set globally.")
        print("     Proceed to next section")
        print("=" * 70)

apply_btn.on_click(on_apply)

# ── Assemble full UI ───────────────────────────────────────────

header = widgets.HTML("""
<div style="background:#f0f4ff; padding:14px; border-left:5px solid #3a6bbf;
            border-radius:4px; margin-bottom:12px;">
  <h3 style="margin:0 0 6px 0; color:#1a3a70;">
    Moral Weight Specification Panel
  </h3>
  <p style="margin:0; font-size:0.92em; color:#333;">
    Declare your credences over the 22 ethical frameworks <i>before</i>
    CIVI is computed. This is a normative pre-commitment, not a model parameter.
    Choose a <b>preset</b> for standard configurations, or use
    <b>custom sliders</b> for bespoke allocation.
    Weights are automatically normalised to sum to 1.
  </p>
</div>
""")

full_ui = widgets.VBox([
    header,
    mode_selector,
    preset_box,
    custom_box,
    apply_btn,
    output_area,
])

display(full_ui)

In [ ]:
# ─────────────────────────────────────────────────────────────
# Exponential Benchmark
# Purpose  : Compute the single-rate exponential comparator
#            used throughout later sections as the baseline
#            against which CIVI is evaluated.
# This cell does not need analyst intervention.
# ─────────────────────────────────────────────────────────────

# Standard Ramsey rate
r_ramsey = delta_base + eta_base * g_base
D_exponential = np.exp(-r_ramsey * t)

print("Exponential benchmark computed.")
print(f"  Ramsey rate r = δ + η·g = "
      f"{delta_base} + {eta_base}×{g_base} = {r_ramsey:.4f}")
print(f"  D_exp(0)   = {D_exponential[0]:.6f}  ✓")
print(f"  D_exp(50)  = {D_exponential[50]:.6f}")
print(f"  D_exp(100) = {D_exponential[100]:.6f}")
print(f"  D_exp(300) = {D_exponential[300]:.6f}")
print()
print("─" * 50)
print("  Section 4 complete. Global objects set:")
print(f"  • D_array         shape {D_array.shape}")
print(f"  • CIVI_baseline   shape {CIVI_baseline.shape}  (run Cell 4.3 first)")
print(f"  • D_exponential   shape {D_exponential.shape}")
print(f"  • w_active        sum = {w_active.sum():.6f}")
print("─" * 50)

# Deterministic Analysis & Visualisations

This section establishes the baseline behaviour of CIVI through four analytical figures,
each targeting a distinct formal property established in Section 3.4 of the paper.



***Note: All figures use the baseline parameter specification and active weight vector from the previous section .
No stochastic draws are made here — uncertainty quantification follows in a later section.***

In [ ]:
import matplotlib.patheffects as pe
from matplotlib.lines import Line2D
import matplotlib.cm as cm

function_labels = [f"D{i+1}" for i in range(D_array.shape[0])]

fig, ax = plt.subplots(figsize=(14, 7))

#  Colour palette
palette = [cm.tab20(i % 20) for i in range(20)] + \
          [cm.Set1(0), cm.Set1(1)]

#  Individual functions (background)
for i in range(22):
    ax.plot(t, D_array[i], color=palette[i],
            linewidth=0.9, alpha=0.32, zorder=1)

#  Shaded envelope
D_min = D_array.min(axis=0)
D_max = D_array.max(axis=0)
ax.fill_between(t, D_min, D_max, alpha=0.07, color="#4a90d9", zorder=1)

#  Exponential benchmark
ax.plot(t, D_exponential, color="#d62728", linewidth=2.4,
        linestyle="--", zorder=3)

#  CIVI composite
ax.plot(t, CIVI_baseline, color="#1a5fa8", linewidth=3.0, zorder=4)


RIGHT_T = 295

endpoint_items = [
    ("CIVI composite", CIVI_baseline[RIGHT_T], "#1a5fa8",  True,  3.0),
    (f"Exponential  r={r_ramsey:.4f}",
     D_exponential[RIGHT_T], "#d62728", True, 2.4),
]

val_at_300 = D_array[:, 300]
idx_max = int(np.argmax(val_at_300))
idx_min = int(np.argmin(val_at_300))

if abs(val_at_300[idx_max] - CIVI_baseline[300]) > 0.02:
    endpoint_items.append(
        (f"{function_labels[idx_max]} (slowest)",
         D_array[idx_max, RIGHT_T], palette[idx_max], False, 0.9)
    )
if abs(val_at_300[idx_min] - D_exponential[300]) > 0.02:
    endpoint_items.append(
        (f"{function_labels[idx_min]} (fastest)",
         D_array[idx_min, RIGHT_T], palette[idx_min], False, 0.9)
    )

endpoint_items.sort(key=lambda x: x[1], reverse=True)

MIN_GAP   = 0.045
label_ys  = []

for label, raw_y, colour, bold, lw in endpoint_items:
    # Push y down if too close to an already-placed label
    placed_y = raw_y
    for prev_y in label_ys:
        if abs(placed_y - prev_y) < MIN_GAP:
            placed_y = prev_y - MIN_GAP

    placed_y = max(placed_y, 0.0)   # never below axis
    label_ys.append(placed_y)

    ax.scatter([RIGHT_T], [raw_y], color=colour, s=28,
               zorder=6, clip_on=True)

    if abs(placed_y - raw_y) > 0.005:
        ax.annotate(
            "", xy=(RIGHT_T, raw_y), xytext=(305, placed_y),
            arrowprops=dict(arrowstyle="-", color=colour,
                            lw=0.8, alpha=0.6),
            annotation_clip=False,
        )

    weight = "bold" if bold else "normal"
    txt = ax.text(
        307, placed_y, label,
        fontsize=8.2, color=colour, fontweight=weight,
        va="center", ha="left",
        transform=ax.transData,
        path_effects=[pe.withStroke(linewidth=2, foreground="white")],
    )
    txt.set_clip_on(False)   # equivalent to annotation_clip=False, valid on Text

# legend
legend_handles = [
    Line2D([0], [0], color="#1a5fa8", linewidth=3.0,
           label="CIVI composite (active weights)"),
    Line2D([0], [0], color="#d62728", linewidth=2.4, linestyle="--",
           label=f"Exponential benchmark  (r = {r_ramsey:.4f})"),
    Line2D([0], [0], color="grey", linewidth=1.0, alpha=0.5,
           label="Individual discount functions (D₁–D₂₂)"),
    plt.Rectangle((0,0), 1, 1, fc="#4a90d9", alpha=0.15, ec="none",
                  label="Ethical framework envelope (min–max)"),
]
ax.legend(handles=legend_handles, fontsize=9.5, loc="lower left",
          framealpha=0.93, edgecolor="#cccccc")

# Formatting
ax.set_xlim(0, 300)
ax.set_ylim(-0.02, 1.08)
ax.set_xlabel("Time horizon (years)", fontsize=12)
ax.set_ylabel("Discount factor  D(t)", fontsize=12)
ax.set_title(
    "Figure 1 — Discount Factor Trajectories:\n"
    "Individual Functions, CIVI Composite, and Exponential Benchmark",
    fontsize=13, fontweight="bold", pad=14,
)
ax.grid(True, alpha=0.3, linewidth=0.6)
ax.spines[["top", "right"]].set_visible(False)

plt.subplots_adjust(right=0.78)

plt.savefig("fig1_discount_factor_trajectories.png", dpi=150,
            bbox_inches="tight")
plt.show()

print("Figure 1 saved.")
print()

civi_range = CIVI_baseline.max() - CIVI_baseline.min()
if civi_range < 0.01:
    print("⚠  WARNING: CIVI_baseline is nearly flat (range = "
          f"{civi_range:.6f}).")
    print("   This means most discount functions are returning 1.0.")
    print("   ROOT CAUSE: parameter names in Cell 4.2 param_kwargs likely")
    print("   don't match the argument names in your Section 3 function")
    print("   definitions. Check that each d_i() function accepts the")
    print("   keywords passed in param_kwargs, or switch to positional args.")
    print()
else:
    print(f"  CIVI range over [0,300]: {civi_range:.4f}  ✓")
    print(f"  CIVI(300) = {CIVI_baseline[300]:.5f}")
    print(f"  D_exp(300) = {D_exponential[300]:.5f}")
    print(f"  Gap: {CIVI_baseline[300] - D_exponential[300]:+.5f}")

In [ ]:
# Log-derivative: r_eff(t) = -d/dt ln CIVI(t)
ln_CIVI    = np.log(np.maximum(CIVI_baseline, 1e-12))
r_eff_CIVI = -np.gradient(ln_CIVI, t)

# Exponential rate is constant
r_eff_exp  = np.full(len(t), r_ramsey)

# Per-function effective rates
r_eff_each = np.zeros((22, len(t)))
for i in range(22):
    ln_Di          = np.log(np.maximum(D_array[i], 1e-12))
    r_eff_each[i]  = -np.gradient(ln_Di, t)

# Gap
gap = r_eff_CIVI - r_eff_exp

print("Effective rate diagnostics")
print(f"  Exponential (constant) : {r_ramsey:.5f}")
print(f"  CIVI r_eff at t=0      : {r_eff_CIVI[0]:.5f}")
print(f"  CIVI r_eff at t=100    : {r_eff_CIVI[100]:.5f}")
print(f"  CIVI r_eff at t=300    : {r_eff_CIVI[300]:.5f}")
declining = r_eff_CIVI[300] < r_eff_CIVI[0]
print(f"  Declining rate property: {'✓ CONFIRMED' if declining else '✗ NOT CONFIRMED'}")

In [ ]:
r1 = delta_base + eta_base * g_base

D_fast   = np.exp(-0.06  * t)
D_mod    = np.exp(-r1    * t)
D_slow   = np.exp(-0.015 * t)
D_hyp    = 1.0 / (1.0 + 0.03 * t)
D_lt     = np.exp(-0.005 * t)

D_dual   = np.where(
    t <= 30,
    np.exp(-0.07 * t),
    np.exp(-0.07 * 30) * np.exp(-0.02 * (t - 30))
)

local_funcs = [D_fast, D_mod, D_slow, D_hyp, D_lt, D_dual]
local_names = ["Fast exp (r=0.06)", "Standard Ramsey (r=0.045)",
               "Slow exp (r=0.015)", "Hyperbolic (k=0.03)",
               "Longtermist (r=0.005)", "Dual-rate (0.07→0.02)"]
local_cols  = ["#d62728", "#e07b39", "#f0c94a",
               "#74c476", "#2166ac", "#9b59b6"]

w_local   = np.ones(6) / 6
D_local   = np.vstack(local_funcs)           # shape (6, 301)
CIVI_local = np.dot(w_local, D_local)

def eff_rate(D):
    return -np.gradient(np.log(np.maximum(D, 1e-12)), t)

r_civi_local = eff_rate(CIVI_local)
r_exp_const  = np.full(len(t), r1)
gap_local    = r_civi_local - r_exp_const

#  Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
fig.patch.set_facecolor("#fafafa")

BLUE  = "#2166ac"
RED   = "#d62728"
GREEN = "#2ca02c"
GREY  = "#888888"

ax = axes[0]
ax.set_facecolor("#f7f9fc")

for D, name, c in zip(local_funcs, local_names, local_cols):
    ax.plot(t, eff_rate(D), color=c, linewidth=1.4,
            alpha=0.55, linestyle="--", label=name)

ax.axhline(r1, color=RED, linewidth=2.2, linestyle=":",
           label=f"Exponential benchmark ({r1:.3f})", zorder=3)
ax.plot(t, r_civi_local, color=BLUE, linewidth=2.8,
        label="CIVI composite rate", zorder=4)

ax.fill_between(t, r_civi_local, r_exp_const,
                where=(r_civi_local < r_exp_const),
                color=GREEN, alpha=0.18, interpolate=True)
ax.fill_between(t, r_civi_local, r_exp_const,
                where=(r_civi_local > r_exp_const),
                color=RED, alpha=0.12, interpolate=True)

ax.set_xlim(0, 300)
ax.set_ylim(bottom=0)
ax.set_xlabel("Years", fontsize=11)
ax.set_ylabel("Instantaneous discount rate", fontsize=11)
ax.set_title("Component Rates and CIVI Composite Rate",
             fontsize=11, fontweight="bold", pad=10)
ax.legend(fontsize=8, framealpha=0.95, edgecolor="#dddddd",
          loc="upper right")
ax.grid(True, alpha=0.3, linewidth=0.5)
ax.spines[["top", "right"]].set_visible(False)

ax2 = axes[1]
ax2.set_facecolor("#f7f9fc")
ax2.axhline(0, color=GREY, linewidth=1.0, linestyle="--", zorder=2)

ax2.fill_between(t, gap_local, 0,
                 where=(gap_local < 0), color=GREEN, alpha=0.30,
                 interpolate=True, label="CIVI rate < Exponential")
ax2.fill_between(t, gap_local, 0,
                 where=(gap_local > 0), color=RED, alpha=0.20,
                 interpolate=True, label="CIVI rate > Exponential")
ax2.plot(t, gap_local, color=BLUE, linewidth=2.0, zorder=3)

for yr in [30, 100, 200, 300]:
    ax2.text(yr, gap_local[yr] - 0.002,
             f"t={yr}\n{gap_local[yr]:+.4f}",
             ha="center", fontsize=7.5, color="#444444")

ax2.set_xlim(0, 300)
ax2.set_xlabel("Years", fontsize=11)
ax2.set_ylabel("r_CIVI(t) − r_exponential", fontsize=11)
ax2.set_title("Rate Divergence: CIVI minus Exponential\n"
              "Negative = CIVI assigns more weight to far future",
              fontsize=11, fontweight="bold", pad=10)
ax2.legend(fontsize=9, framealpha=0.95, edgecolor="#dddddd")
ax2.grid(True, alpha=0.3, linewidth=0.5)
ax2.spines[["top", "right"]].set_visible(False)

fig.suptitle(
    "CIVI vs Exponential Benchmark",
    fontsize=13, fontweight="bold", y=1.02)

plt.tight_layout()
plt.savefig("fig2_effective_rate.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure 2 saved.")

In [ ]:
#Pair definitions

# Pair 1: Quasi-hyperbolic vs Exponential
beta_qh = 0.75
r_qh    = 0.01     # long-run rate
r_std   = 0.03     # standard rate
D_qh    = np.where(t == 0, 1.0, beta_qh * np.exp(-r_qh * t))
D_std   = np.exp(-r_std * t)

# Pair 2: Dual-rate vs Moderate exponential
D_dual2 = np.where(
    t <= 30,
    np.exp(-0.07 * t),
    np.exp(-0.07 * 30) * np.exp(-0.02 * (t - 30))
)
D_mod2  = np.exp(-0.04 * t)

# Pair 3: Gen. hyperbolic steep vs gen. hyperbolic flat
D_gh_steep = (1 + 0.01 * t) ** (-2.0)   # drops fast, then flattens
D_gh_flat  = (1 + 0.05 * t) ** (-0.5)   # drops moderately

# Pair 4: Catastrophic risk vs Longtermist
r_cat  = delta_base + eta_base * g_base + p_catastrophe * L_loss
D_cat  = np.exp(-r_cat * t)
D_lt2  = np.exp(-0.003 * t)

# Pair 5: Precautionary vs Standard (likely no crossover)
r_prec = delta_base + eta_base * g_base \
         - 0.5 * eta_base * (eta_base + 1) * sigma_g**2
D_prec = np.exp(-r_prec * t)
D_ram  = np.exp(-(delta_base + eta_base * g_base) * t)

pairs_fig3 = [
    (D_qh,        D_std,      "Quasi-hyperbolic (β=0.75)",  "Exponential (r=0.03)",       "#e05c39", "#1a6bb5"),
    (D_dual2,     D_mod2,     "Dual-rate (0.07→0.02)",      "Moderate exp (r=0.04)",      "#9b59b6", "#2a7a3b"),
    (D_gh_steep,  D_gh_flat,  "Gen. hyperbolic (α=2)",      "Gen. hyperbolic (α=0.5)",    "#d4a017", "#c0392b"),
    (D_cat,       D_lt2,      "Catastrophic risk",          "Longtermist (r=0.003)",      "#c0392b", "#2166ac"),
    (D_prec,      D_ram,      "Precautionary savings",      "Standard Ramsey",            "#27ae60", "#e07b39"),
]

def find_crossover(Di, Dj):
    diff = Di - Dj
    changes = np.where(np.diff(np.sign(diff)))[0]
    if len(changes) == 0:
        return None
    idx  = changes[0]
    frac = -diff[idx] / (diff[idx+1] - diff[idx] + 1e-12)
    return float(t[idx] + frac)

#  Figure
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.patch.set_facecolor("#fafafa")
axes = axes.flatten()

for idx, (Di, Dj, li, lj, c1, c2) in enumerate(pairs_fig3):
    ax = axes[idx]
    ax.set_facecolor("#f7f9fc")

    ax.plot(t, Di, color=c1, linewidth=2.2, label=li)
    ax.plot(t, Dj, color=c2, linewidth=2.2, linestyle="--", label=lj)

    ax.fill_between(t, Di, Dj,
                    where=(Di >= Dj), alpha=0.10, color=c1,
                    interpolate=True)
    ax.fill_between(t, Di, Dj,
                    where=(Di <  Dj), alpha=0.10, color=c2,
                    interpolate=True)

    cx = find_crossover(Di, Dj)

    if cx is not None:
        cx_val = float(np.interp(cx, t, Di))
        ax.axvline(cx, color="#555555", linewidth=1.4,
                   linestyle=":", zorder=4)
        ax.scatter([cx], [cx_val], color="#222222",
                   s=60, zorder=5)
        # Place label left or right depending on space
        x_off = cx + 8 if cx < 230 else cx - 55
        ax.text(x_off, min(cx_val + 0.06, 0.95),
                f"t ≈ {cx:.0f}",
                fontsize=9.5, color="#222222", fontweight="bold")
    else:
        ax.text(150, 0.55,
                "No crossover\n(rate divergence\nincreases monotonically)",
                ha="center", va="center", fontsize=8.5,
                color="#888888", style="italic",
                bbox=dict(boxstyle="round,pad=0.3",
                          facecolor="white", edgecolor="#cccccc",
                          alpha=0.85))

    ax.set_xlim(0, 300)
    ax.set_ylim(0, 1.08)
    ax.set_xlabel("Years", fontsize=9.5)
    ax.set_ylabel("Discount factor", fontsize=9.5)
    ax.set_title(f"{li}  vs\n{lj}", fontsize=9.5, fontweight="bold")
    ax.legend(fontsize=8, loc="upper right",
              framealpha=0.93, edgecolor="#dddddd")
    ax.grid(True, alpha=0.3, linewidth=0.5)
    ax.spines[["top", "right"]].set_visible(False)

# 6th panel — summary table
ax6 = axes[5]
ax6.set_facecolor("#f0f4ff")
ax6.axis("off")

lines = ["Crossover Summary", "─" * 26]
for Di, Dj, li, lj, c1, c2 in pairs_fig3:
    cx = find_crossover(Di, Dj)
    cx_str = f"t ≈ {cx:.0f} years" if cx is not None else "None in [0, 300]"
    lines.append(f"{li[:22]}\n  vs {lj[:22]}\n  → {cx_str}\n")

lines += ["─" * 26,
          "Crossover = horizon at which",
          "relative framework rankings",
          "invert. Policy implications",
          "are horizon-dependent."]

ax6.text(0.06, 0.96, "\n".join(lines),
         transform=ax6.transAxes, fontsize=8.2,
         verticalalignment="top", fontfamily="monospace",
         bbox=dict(boxstyle="round,pad=0.5", facecolor="white",
                   edgecolor="#aaaaaa", alpha=0.92))

fig.suptitle(
    "Crossover Dynamics Between Competing Ethical Discount Functions",
    fontsize=13, fontweight="bold", y=1.01)

plt.tight_layout()
plt.savefig("fig3_crossover_dynamics.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure 3 saved.")
print()
print("Crossover years:")
for Di, Dj, li, lj, *_ in pairs_fig3:
    cx = find_crossover(Di, Dj)
    cx_str = f"t ≈ {cx:.0f}" if cx is not None else "None"
    print(f"  {li:<30} vs {lj:<25} → {cx_str}")

# Policy Application of CIVI - Tonne-Year Accounting

**Tonne-year accounting** credits CDR projects for the *duration* of carbon storage,
not just the quantity. One **tonne**-year represents one tonne of CO₂ stored for one year.

A project storing $Q$ tonnes for $T_{store}$ years generates a uniform annual
benefit stream:

$$X_{TY}(t) = \mathbf{1}_{[0,\, T_{store}]}(t)$$

where $\mathbf{1}$ is the indicator function equal to 1 when $t \leq T_{store}$,
zero thereafter. The present value of that stream under any discount function $D(t)$ is:

$$PV = \sum_{t=0}^{T_{store}} D(t) \cdot X_{TY}(t) = \sum_{t=0}^{T_{store}} D(t)$$

Under **CIVI**:

$$PV_{CIVI} = \sum_{t=0}^{T_{store}} CIVI(t) \cdot 1$$

Under the **exponential benchmark**:

$$PV_{exp} = \sum_{t=0}^{T_{store}} e^{-rt} \cdot 1 = \frac{1 - e^{-r(T_{store}+1)}}{1 - e^{-r}}$$



| Project | Technology | Storage duration |
|---|---|---|
| Forestry | Afforestation / reforestation | 30 years |
| Biochar | Soil carbon sequestration | 100 years |
| DAC | Direct air capture + geological storage | 300 years |

In [ ]:
# Storage durations
T_forestry = 30
T_biochar  = 100
T_DAC      = 300

# Benefit streams: 1 tonne-year per year during storage, 0 after
X_forestry = np.where(t <= T_forestry, 1.0, 0.0)
X_biochar  = np.where(t <= T_biochar,  1.0, 0.0)
X_DAC      = np.where(t <= T_DAC,      1.0, 0.0)

streams = {
    "Forestry (30yr)":  (X_forestry, T_forestry, "#2a7a3b"),
    "Biochar (100yr)":  (X_biochar,  T_biochar,  "#d4a017"),
    "DAC (300yr)":      (X_DAC,      T_DAC,      "#2166ac"),
}

print("Ton-year benefit streams constructed")
print(f"  t array: 0 to {int(t[-1])} ({len(t)} steps)")
print()
print(f"  {'Project':<22} {'T_store':>8} {'Total ton-years':>16}")
print("  " + "─" * 48)
for name, (X, T_s, c) in streams.items():
    print(f"  {name:<22} {T_s:>8}     {int(X.sum()):>12}")

In [ ]:
# PV calculation

D_civi = CIVI_local
D_exp  = D_exponential

results = {}

for name, (X, T_s, col) in streams.items():
    PV_civi      = float(np.sum(D_civi * X))
    PV_exp       = float(np.sum(D_exp  * X))
    cumPV_civi   = np.cumsum(D_civi * X)
    cumPV_exp    = np.cumsum(D_exp  * X)

    results[name] = {
        "T_store":    T_s,
        "colour":     col,
        "X":          X,
        "PV_civi":    PV_civi,
        "PV_exp":     PV_exp,
        "PV_ratio":   PV_civi / PV_exp if PV_exp > 0 else np.nan,
        "cumPV_civi": cumPV_civi,
        "cumPV_exp":  cumPV_exp,
    }

#  Metric 1: Absolute PV
print("=" * 68)
print("  METRIC 1 — Absolute Present Value")
print("=" * 68)
print(f"  {'Project':<22} {'PV (CIVI)':>12} {'PV (Exp)':>12} {'CIVI Premium':>14}")
print("  " + "─" * 62)
for name, r in results.items():
    premium = r["PV_civi"] - r["PV_exp"]
    print(f"  {name:<22} {r['PV_civi']:>12.3f} {r['PV_exp']:>12.3f} "
          f"{premium:>+14.3f}")
print()
print("  Note: absolute rankings are the same under both methods.")
print("  Longer storage always accumulates more undiscounted years.")
print("  The interesting story is in relative value — see below.")

#  Metric 2: Value index (Forestry = 1.0)
base_civi = results["Forestry (30yr)"]["PV_civi"]
base_exp  = results["Forestry (30yr)"]["PV_exp"]

print()
print("=" * 68)
print("  METRIC 2 — Value Index  (Forestry = 1.00)")
print("  How much more valuable is each project RELATIVE to Forestry?")
print("=" * 68)
print(f"  {'Project':<22} {'Index CIVI':>12} {'Index Exp':>12}  {'Reversal?':>12}")
print("  " + "─" * 62)

idx_civi_vals = {}
idx_exp_vals  = {}

for name, r in results.items():
    idx_civi = r["PV_civi"] / base_civi
    idx_exp  = r["PV_exp"]  / base_exp
    idx_civi_vals[name] = idx_civi
    idx_exp_vals[name]  = idx_exp

    gap = idx_civi - idx_exp
    flag = f"↑ +{gap:.2f}" if gap > 0.05 else "—"
    print(f"  {name:<22} {idx_civi:>12.3f} {idx_exp:>12.3f}  {flag:>12}")

print()
print("  KEY FINDING:")
print("  Under exponential discounting, DAC is barely more valuable than")
print("  Forestry (years 31-300 worth ~0). Under CIVI, DAC is substantially")
print("  more valuable — CIVI preserves the premium for permanent storage.")

#  Metric 3: CIVI premium by project
print()
print("=" * 68)
print("  METRIC 3 — CIVI Premium Ratio  (PV_CIVI / PV_Exp)")
print("  Shows which project benefits MOST from choosing CIVI over Exp")
print("=" * 68)
print(f"  {'Project':<22} {'Ratio':>10}  {'Interpretation':>30}")
print("  " + "─" * 66)

ratios = {}
for name, r in results.items():
    ratio = r["PV_ratio"]
    ratios[name] = ratio
    uplift = (ratio - 1) * 100
    interp = f"CIVI gives {uplift:+.1f}% more value"
    print(f"  {name:<22} {ratio:>10.4f}  {interp:>30}")

print()
print("  CONCLUSION: The CIVI premium grows with storage duration.")
print("  Exponential discounting compresses the advantage of permanence.")
print("  CIVI restores it — making long-duration CDR relatively more")
print("  competitive under ethical multi-framework aggregation.")

#  Rankings
print()
print("=" * 68)
print("  RANKINGS")
print("=" * 68)

rank_abs_civi = sorted(results.keys(),
                       key=lambda k: results[k]["PV_civi"], reverse=True)
rank_abs_exp  = sorted(results.keys(),
                       key=lambda k: results[k]["PV_exp"],  reverse=True)
rank_idx_civi = sorted(results.keys(),
                       key=lambda k: idx_civi_vals[k], reverse=True)
rank_idx_exp  = sorted(results.keys(),
                       key=lambda k: idx_exp_vals[k],  reverse=True)

print(f"\n  Absolute PV ranking (same under both — expected):")
for i, (rc, re) in enumerate(zip(rank_abs_civi, rank_abs_exp), 1):
    print(f"    {i}. CIVI: {rc:<25}  Exp: {re}")

print(f"\n  Relative value index ranking (Forestry = base):")
print(f"  {'Rank':<6} {'CIVI index order':<30} {'Exp index order':<30} {'Same?'}")
print("  " + "─" * 72)
for i, (rc, re) in enumerate(zip(rank_idx_civi, rank_idx_exp), 1):
    same = "✓" if rc == re else "✗ REVERSAL"
    print(f"  {i:<6} {rc:<30} {re:<30} {same}")

#  Effective horizon diagnostic
print()
print("=" * 68)
print("  DIAGNOSTIC — Effective Discount Factor at Key Horizons")
print("  (Shows why CIVI treats long-duration storage differently)")
print("=" * 68)
print(f"  {'Year':<8} {'CIVI(t)':>12} {'Exp(t)':>12}  {'Ratio':>10}")
print("  " + "─" * 46)
for yr in [30, 50, 100, 150, 200, 300]:
    c  = float(D_civi[yr])
    e  = float(D_exp[yr])
    rt = c / e if e > 1e-8 else float("inf")
    print(f"  {yr:<8} {c:>12.5f} {e:>12.5f}  {rt:>10.2f}×")

print()
print("  At t=100, CIVI retains ~16× more value than exponential.")
print("  At t=200, the gap widens further — this is what makes DAC")
print("  substantially more competitive under CIVI discounting.")

In [ ]:
#Cumulative PV

fig, axes = plt.subplots(1, 3, figsize=(16, 5.5))
fig.patch.set_facecolor("#fafafa")

for ax, (name, r) in zip(axes, results.items()):
    ax.set_facecolor("#f7f9fc")

    T_s        = r["T_store"]
    civi_curve = r["cumPV_civi"]
    exp_curve  = r["cumPV_exp"]
    col        = r["colour"]

    # Shade the gap
    ax.fill_between(t, civi_curve, exp_curve,
                    where=(civi_curve >= exp_curve),
                    alpha=0.20, color=col, interpolate=True)
    ax.fill_between(t, civi_curve, exp_curve,
                    where=(civi_curve < exp_curve),
                    alpha=0.20, color="#d62728", interpolate=True)

    # Main curves
    ax.plot(t, civi_curve, color=col, linewidth=2.6,
            label=f"CIVI  (PV = {r['PV_civi']:.2f})")
    ax.plot(t, exp_curve,  color="#d62728", linewidth=2.0,
            linestyle="--", label=f"Exp  (PV = {r['PV_exp']:.2f})")

    # Storage cutoff
    ax.axvline(T_s, color="#888888", linewidth=1.2, linestyle=":")
    ax.text(T_s + 3, 0.15, f"T = {T_s}yr",
            fontsize=8, color="#666666")

    # Gap annotation at end
    gap = r["PV_civi"] - r["PV_exp"]
    ax.annotate(
        f"Gap = {gap:.2f}",
        xy=(300, (r["PV_civi"] + r["PV_exp"]) / 2),
        fontsize=8.5, color="#333333", ha="right",
        bbox=dict(boxstyle="round,pad=0.3", facecolor="white",
                  edgecolor="#cccccc", alpha=0.9)
    )

    ax.set_xlim(0, 300)
    ax.set_ylim(bottom=0)
    ax.set_xlabel("Years", fontsize=11)
    ax.set_ylabel("Cumulative discounted ton-years", fontsize=10)
    ax.set_title(name, fontsize=12, fontweight="bold", pad=10)
    ax.legend(fontsize=8.5, framealpha=0.95, edgecolor="#dddddd",
              loc="lower right")
    ax.grid(True, alpha=0.3, linewidth=0.5)
    ax.spines[["top", "right"]].set_visible(False)

fig.suptitle(
    "Cumulative Present Value of Ton-Year Streams: "
    "CIVI vs Exponential Discounting\n"
    "Shaded area = valuation differential attributable to ethical "
    "framework choice",
    fontsize=12, fontweight="bold", y=1.03)

plt.tight_layout()
plt.savefig("fig5_cumulative_PV.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure 5 saved.")

In [ ]:
project_names = list(results.keys())
short_names   = ["Forestry\n(30yr)", "Biochar\n(100yr)", "DAC\n(300yr)"]
PV_civi_vals  = [results[n]["PV_civi"]   for n in project_names]
PV_exp_vals   = [results[n]["PV_exp"]    for n in project_names]
proj_cols     = [results[n]["colour"]    for n in project_names]
ratio_vals    = [results[n]["PV_ratio"]  for n in project_names]
uplift_vals   = [(r - 1) * 100 for r in ratio_vals]

# Value index (Forestry = 1.0)
idx_civi = [idx_civi_vals[n] for n in project_names]
idx_exp  = [idx_exp_vals[n]  for n in project_names]

fig, axes = plt.subplots(1, 3, figsize=(17, 5.5))
fig.patch.set_facecolor("#fafafa")

x      = np.arange(3)
w      = 0.35
BLUE   = "#2166ac"
RED    = "#d62728"
GREY   = "#888888"

ax = axes[0]
ax.set_facecolor("#f7f9fc")

b1 = ax.bar(x - w/2, PV_exp_vals,  width=w, color=RED,
            alpha=0.75, label="Exponential",
            edgecolor="white", linewidth=0.8)
b2 = ax.bar(x + w/2, PV_civi_vals, width=w, color=proj_cols,
            alpha=0.88, label="CIVI",
            edgecolor="white", linewidth=0.8)

for bar, val in zip(b1, PV_exp_vals):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.3,
            f"{val:.1f}", ha="center", fontsize=8, color=RED,
            fontweight="bold")
for bar, val in zip(b2, PV_civi_vals):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.3,
            f"{val:.1f}", ha="center", fontsize=8, color="#1a3a70",
            fontweight="bold")

ax.set_xticks(x)
ax.set_xticklabels(short_names, fontsize=10)
ax.set_ylabel("Present Value (ton-years)", fontsize=10)
ax.set_title("A — Absolute PV\n(rankings same — both rise with duration)",
             fontsize=10, fontweight="bold", pad=8)
ax.legend(fontsize=9, framealpha=0.95, edgecolor="#dddddd")
ax.grid(True, axis="y", alpha=0.3, linewidth=0.5)
ax.spines[["top", "right"]].set_visible(False)

ax2 = axes[1]
ax2.set_facecolor("#f7f9fc")

b3 = ax2.bar(x - w/2, idx_exp,  width=w, color=RED,
             alpha=0.75, label="Exponential",
             edgecolor="white", linewidth=0.8)
b4 = ax2.bar(x + w/2, idx_civi, width=w, color=proj_cols,
             alpha=0.88, label="CIVI",
             edgecolor="white", linewidth=0.8)

ax2.axhline(1.0, color=GREY, linewidth=1.2, linestyle="--",
            label="Forestry baseline")

for bar, val in zip(b3, idx_exp):
    ax2.text(bar.get_x() + bar.get_width()/2, val + 0.02,
             f"{val:.2f}×", ha="center", fontsize=8,
             color=RED, fontweight="bold")
for bar, val in zip(b4, idx_civi):
    ax2.text(bar.get_x() + bar.get_width()/2, val + 0.02,
             f"{val:.2f}×", ha="center", fontsize=8,
             color="#1a3a70", fontweight="bold")

# Arrow highlighting DAC divergence
dac_idx = 2
ax2.annotate(
    "",
    xy=(dac_idx + w/2,  idx_civi[dac_idx]),
    xytext=(dac_idx - w/2, idx_exp[dac_idx]),
    arrowprops=dict(arrowstyle="<->", color="#333333",
                    lw=1.8, mutation_scale=14)
)
gap_dac = idx_civi[dac_idx] - idx_exp[dac_idx]
ax2.text(dac_idx + 0.05,
         (idx_civi[dac_idx] + idx_exp[dac_idx]) / 2,
         f"Δ = {gap_dac:.2f}×",
         fontsize=8.5, color="#333333", fontweight="bold",
         bbox=dict(boxstyle="round,pad=0.25", facecolor="white",
                   edgecolor="#aaaaaa", alpha=0.9))

ax2.set_xticks(x)
ax2.set_xticklabels(short_names, fontsize=10)
ax2.set_ylabel("Value relative to Forestry (= 1.0)", fontsize=10)
ax2.set_title("B — Relative Value Index\n"
              "(CIVI premium for permanence is structurally higher)",
              fontsize=10, fontweight="bold", pad=8)
ax2.legend(fontsize=9, framealpha=0.95, edgecolor="#dddddd")
ax2.grid(True, axis="y", alpha=0.3, linewidth=0.5)
ax2.spines[["top", "right"]].set_visible(False)

# ── PANEL C: CIVI uplift % ─────────────────────────────────────
ax3 = axes[2]
ax3.set_facecolor("#f7f9fc")

bar_cols = ["#2ca02c" if u >= 0 else RED for u in uplift_vals]
b5 = ax3.bar(x, uplift_vals, width=0.50, color=bar_cols,
             alpha=0.85, edgecolor="white", linewidth=0.8)

ax3.axhline(0, color=GREY, linewidth=1.2, linestyle="--")

for bar, val in zip(b5, uplift_vals):
    ypos = val + 0.4 if val >= 0 else val - 1.5
    ax3.text(bar.get_x() + bar.get_width()/2, ypos,
             f"+{val:.1f}%" if val >= 0 else f"{val:.1f}%",
             ha="center", fontsize=9.5, fontweight="bold",
             color="#1a3a70")

ax3.set_xticks(x)
ax3.set_xticklabels(short_names, fontsize=10)
ax3.set_ylabel("CIVI uplift over Exponential (%)", fontsize=10)
ax3.set_title("C — CIVI Premium by Project\n"
              "(Premium grows with storage duration)",
              fontsize=10, fontweight="bold", pad=8)
ax3.grid(True, axis="y", alpha=0.3, linewidth=0.5)
ax3.spines[["top", "right"]].set_visible(False)

fig.suptitle(
    "Comparative Present Value of CDR Projects Under "
    "Alternative Discount Frameworks\n"
    "Panel B is the key result: CIVI structurally favours "
    "long-duration storage relative to exponential discounting",
    fontsize=11, fontweight="bold", y=1.03)

plt.tight_layout()
plt.savefig("fig6_CDR_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure 6 saved.")

# Uncertainty: Monte Carlo Simulation

Monte Carlo propagates *empirical* parameter uncertainty through CIVI,
producing a **distribution** of present values rather than a single point estimate.

$$\delta \sim \text{TruncNormal}(\mu=0.015,\, \sigma=0.007,\, [0, 0.03])$$
$$\eta \sim \text{Normal}(\mu=1.5,\, \sigma=0.5) \text{ clipped to } [0.5, 4.0]$$
$$g \sim \text{Normal}(\mu=0.02,\, \sigma=0.01) \text{ clipped to } [0, 0.03]$$


Parameter draws replace baseline values; moral weights remain fixed.

In [ ]:
# parameter distribution
from scipy.stats import truncnorm, norm, beta as beta_dist, lognorm
import warnings
warnings.filterwarnings("ignore")

np.random.seed(42)

N = 10_000   # simulations

# Sampling functions

def sample_delta(n):
    # TruncNormal on [0, 0.03], mean=0.015, sd=0.007
    a, b = (0 - 0.015) / 0.007, (0.03 - 0.015) / 0.007
    return truncnorm.rvs(a, b, loc=0.015, scale=0.007, size=n)

def sample_eta(n):
    return np.clip(norm.rvs(loc=1.5, scale=0.5, size=n), 0.5, 4.0)

def sample_g(n):
    return np.clip(norm.rvs(loc=0.02, scale=0.01, size=n), 0.0, 0.03)

def sample_p_cat(n):
    # Beta(2,200) — small probabilities centred ~0.01
    return beta_dist.rvs(2, 200, size=n)

def sample_L(n):
    # LogNormal centred on 0.3
    return np.clip(lognorm.rvs(s=0.3, scale=0.3, size=n), 0.05, 0.90)

def sample_k(n):
    # Uniform on [0.01, 0.05]
    return np.random.uniform(0.01, 0.05, size=n)

draws = {
    "delta": sample_delta(N),
    "eta":   sample_eta(N),
    "g":     sample_g(N),
    "p_cat": sample_p_cat(N),
    "L":     sample_L(N),
    "k":     sample_k(N),
}

#  Plot distributions
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
fig.patch.set_facecolor("#fafafa")
axes = axes.flatten()

param_info = [
    ("delta", "δ — Pure time preference",    "#2166ac"),
    ("eta",   "η — Marginal utility elast.", "#e05c39"),
    ("g",     "g — Consumption growth",      "#2a7a3b"),
    ("p_cat", "p — Catastrophe probability", "#9b59b6"),
    ("L",     "L — Welfare loss on shock",   "#d4a017"),
    ("k",     "k — Hyperbolic parameter",    "#c0392b"),
]

for ax, (key, label, col) in zip(axes, param_info):
    ax.set_facecolor("#f7f9fc")
    data = draws[key]
    ax.hist(data, bins=60, color=col, alpha=0.75,
            edgecolor="white", linewidth=0.4)
    ax.axvline(data.mean(),   color="#222222", linewidth=1.8,
               linestyle="-",  label=f"Mean = {data.mean():.4f}")
    ax.axvline(np.percentile(data, 5),  color="#555555",
               linewidth=1.2, linestyle="--",
               label=f"5th = {np.percentile(data,5):.4f}")
    ax.axvline(np.percentile(data, 95), color="#555555",
               linewidth=1.2, linestyle=":",
               label=f"95th = {np.percentile(data,95):.4f}")
    ax.set_title(label, fontsize=10, fontweight="bold")
    ax.set_xlabel("Parameter value", fontsize=9)
    ax.set_ylabel("Frequency", fontsize=9)
    ax.legend(fontsize=7.5, framealpha=0.9)
    ax.grid(True, alpha=0.3, linewidth=0.5)
    ax.spines[["top","right"]].set_visible(False)

fig.suptitle("Monte Carlo Parameter Distributions  (N = 10,000)",
             fontsize=12, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("fig7a_param_distributions.png", dpi=150, bbox_inches="tight")
plt.show()

print("Parameter distribution summary:")
print(f"  {'Param':<8} {'Mean':>9} {'Std':>9} {'5th':>9} {'95th':>9}")
print("  " + "─" * 46)
for key, label, _ in param_info:
    d = draws[key]
    print(f"  {key:<8} {d.mean():>9.5f} {d.std():>9.5f} "
          f"{np.percentile(d,5):>9.5f} {np.percentile(d,95):>9.5f}")

In [ ]:
from tqdm.auto import tqdm

# Store scalar PVs
PV_civi_mc = np.zeros(N)
PV_exp_mc  = np.zeros(N)

N_fan      = 2000           # store this many full trajectories
CIVI_traj  = np.zeros((N_fan, len(t)))
exp_traj   = np.zeros((N_fan, len(t)))

X_dac = X_DAC

print(f"Running Monte Carlo simulation  (N = {N:,})...")

for sim in tqdm(range(N)):

    d   = draws["delta"][sim]
    eta = draws["eta"][sim]
    g_  = draws["g"][sim]
    p_  = draws["p_cat"][sim]
    L_  = draws["L"][sim]
    k_  = draws["k"][sim]

    r_sim    = d + eta * g_
    r_lt     = max(d * 0.1, 0.001)          # longtermist: near-zero

    D1  = np.exp(-r_sim * t)                                    # standard
    D2  = np.exp(-(d * 0.1 + eta * g_) * t)                   # near-zero δ
    D3  = 1.0 / (1.0 + k_ * t)                                # hyperbolic
    D4  = np.exp(-(r_sim + p_ * L_) * t)                      # catastrophic
    D5  = np.exp(-r_lt * t)                                    # longtermist
    # Dual-rate
    r_short  = min(r_sim + 0.02, 0.12)
    r_long   = max(r_sim - 0.015, 0.001)
    D6  = np.where(
        t <= 30,
        np.exp(-r_short * t),
        np.exp(-r_short * 30) * np.exp(-r_long * (t - 30))
    )

    #  CIVI (equal weights)
    D_local_sim = np.vstack([D1, D2, D3, D4, D5, D6])
    civi_sim    = D_local_sim.mean(axis=0)

    #  Exponential benchmark
    exp_sim = np.exp(-r_sim * t)

    #  Store PVs
    PV_civi_mc[sim] = float(np.sum(civi_sim * X_dac))
    PV_exp_mc[sim]  = float(np.sum(exp_sim  * X_dac))

    #  Store trajectories for fan chart
    if sim < N_fan:
        CIVI_traj[sim] = civi_sim
        exp_traj[sim]  = exp_sim

print("Simulation complete.")
print()

#  Summary statistics
def summarise(arr, label):
    print(f"  {label}")
    print(f"    Mean   : {arr.mean():.4f}")
    print(f"    Median : {np.median(arr):.4f}")
    print(f"    Std    : {arr.std():.4f}")
    print(f"    5th    : {np.percentile(arr, 5):.4f}")
    print(f"    95th   : {np.percentile(arr,95):.4f}")
    print(f"    CV     : {arr.std()/arr.mean():.4f}")
    print()

print("=" * 50)
print("  MONTE CARLO RESULTS — DAC (300yr) PV")
print("=" * 50)
summarise(PV_civi_mc, "CIVI")
summarise(PV_exp_mc,  "Exponential")

print(f"  CIVI mean PV / Exp mean PV = "
      f"{PV_civi_mc.mean() / PV_exp_mc.mean():.4f}×")
print(f"  CIVI CV < Exp CV?  "
      f"{'YES — CIVI more stable' if PV_civi_mc.std()/PV_civi_mc.mean() < PV_exp_mc.std()/PV_exp_mc.mean() else 'NO'}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
fig.patch.set_facecolor("#fafafa")

BLUE = "#2166ac"
RED  = "#d62728"

ax = axes[0]
ax.set_facecolor("#f7f9fc")

bins = np.linspace(
    min(PV_civi_mc.min(), PV_exp_mc.min()),
    max(PV_civi_mc.max(), PV_exp_mc.max()),
    70
)

ax.hist(PV_exp_mc,  bins=bins, color=RED,  alpha=0.50,
        label="Exponential", edgecolor="white", linewidth=0.3)
ax.hist(PV_civi_mc, bins=bins, color=BLUE, alpha=0.60,
        label="CIVI", edgecolor="white", linewidth=0.3)

# Vertical lines: mean and CI
for arr, col in [(PV_civi_mc, BLUE), (PV_exp_mc, RED)]:
    ax.axvline(arr.mean(), color=col, linewidth=2.2, linestyle="-")
    ax.axvline(np.percentile(arr,  5), color=col,
               linewidth=1.4, linestyle="--", alpha=0.8)
    ax.axvline(np.percentile(arr, 95), color=col,
               linewidth=1.4, linestyle="--", alpha=0.8)

for arr, col, side in [(PV_civi_mc, BLUE, 1), (PV_exp_mc, RED, -1)]:
    ax.text(arr.mean() + side * 0.3,
            ax.get_ylim()[1] * 0.85 if ax.get_ylim()[1] > 0 else 200,
            f"μ={arr.mean():.2f}", color=col, fontsize=9,
            fontweight="bold", ha="center")

ax.set_xlabel("Present Value (discounted tonne-years)", fontsize=11)
ax.set_ylabel("Frequency", fontsize=11)
ax.set_title("PV Distributions — CIVI vs Exponential\n"
             "(DAC 300yr project,  N = 10,000)",
             fontsize=11, fontweight="bold", pad=10)
ax.legend(fontsize=10, framealpha=0.95, edgecolor="#dddddd")
ax.grid(True, alpha=0.3, linewidth=0.5)
ax.spines[["top","right"]].set_visible(False)

ax2 = axes[1]
ax2.set_facecolor("#f7f9fc")

data_both   = [PV_exp_mc, PV_civi_mc]
labels_both = ["Exponential", "CIVI"]
cols_both   = [RED, BLUE]

bp = ax2.boxplot(
    data_both,
    vert=True,
    patch_artist=True,
    widths=0.45,
    medianprops=dict(color="white", linewidth=2.5),
    whiskerprops=dict(linewidth=1.5),
    capprops=dict(linewidth=1.5),
    flierprops=dict(marker="o", markersize=2.5,
                    alpha=0.25, linestyle="none"),
)

for patch, col in zip(bp["boxes"], cols_both):
    patch.set_facecolor(col)
    patch.set_alpha(0.70)

np.random.seed(99)
for pos, arr, col in zip([1, 2], data_both, cols_both):
    jitter = np.random.normal(0, 0.07, size=400)
    sample = np.random.choice(arr, 400, replace=False)
    ax2.scatter(pos + jitter, sample, color=col,
                alpha=0.18, s=8, zorder=2)

# 90% CI brackets
for pos, arr, col in zip([1, 2], data_both, cols_both):
    lo, hi = np.percentile(arr, 5), np.percentile(arr, 95)
    ax2.plot([pos + 0.28, pos + 0.28], [lo, hi],
             color=col, linewidth=2.5, solid_capstyle="round")
    ax2.text(pos + 0.32, (lo + hi) / 2,
             f"90% CI\n[{lo:.1f}, {hi:.1f}]",
             fontsize=7.5, color=col, va="center")

ax2.set_xticks([1, 2])
ax2.set_xticklabels(labels_both, fontsize=11)
ax2.set_ylabel("Present Value (discounted ton-years)", fontsize=11)
ax2.set_title("Distribution Spread Comparison\n"
              "Boxes = IQR,  Line = median,  Dots = sample draws",
              fontsize=11, fontweight="bold", pad=10)
ax2.grid(True, axis="y", alpha=0.3, linewidth=0.5)
ax2.spines[["top","right"]].set_visible(False)

fig.suptitle("Monte Carlo Distribution of Present Values: "
             "CIVI vs Exponential Discounting",
             fontsize=13, fontweight="bold", y=1.02)

plt.tight_layout()
plt.savefig("fig7b_PV_distributions.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure 7 saved.")

In [ ]:
#  Computing percentile bands at each time step
pcts = [5, 25, 50, 75, 95]

civi_bands = {p: np.percentile(CIVI_traj, p, axis=0) for p in pcts}
exp_bands  = {p: np.percentile(exp_traj,  p, axis=0) for p in pcts}

#  Figure
fig, axes = plt.subplots(1, 2, figsize=(16, 6.5))
fig.patch.set_facecolor("#fafafa")

BLUE  = "#2166ac"
RED   = "#d62728"
LBLUE = "#9ecae1"
LRED  = "#fc9272"

for ax, (bands, col, lcol, label) in zip(
    axes,
    [(civi_bands, BLUE, LBLUE, "CIVI"),
     (exp_bands,  RED,  LRED,  "Exponential")]
):
    ax.set_facecolor("#f7f9fc")

    # Outer band: 5–95
    ax.fill_between(t, bands[5], bands[95],
                    color=lcol, alpha=0.25,
                    label="5th–95th percentile")

    # Inner band: 25–75
    ax.fill_between(t, bands[25], bands[75],
                    color=col, alpha=0.30,
                    label="25th–75th percentile (IQR)")

    # Median
    ax.plot(t, bands[50], color=col, linewidth=2.6,
            label="Median", zorder=4)

    # Baseline deterministic curve for reference
    ref = CIVI_local if label == "CIVI" else D_exponential
    ax.plot(t, ref, color=col, linewidth=1.4,
            linestyle="--", alpha=0.65,
            label="Baseline (deterministic)", zorder=3)

    # Annotate band width at key horizons
    for yr, x_off in [(100, 8), (200, 8), (300, -60)]:
        width = bands[95][yr] - bands[5][yr]
        mid   = (bands[95][yr] + bands[5][yr]) / 2
        ax.annotate(
            f"90% band\n= {width:.3f}",
            xy=(yr, bands[95][yr]),
            xytext=(yr + x_off, mid + 0.05),
            fontsize=7.5, color="#333333",
            arrowprops=dict(arrowstyle="-", color="#aaaaaa", lw=0.8),
            bbox=dict(boxstyle="round,pad=0.2", facecolor="white",
                      edgecolor="#cccccc", alpha=0.85)
        )

    ax.set_xlim(0, 300)
    ax.set_ylim(0, 1.08)
    ax.set_xlabel("Years", fontsize=11)
    ax.set_ylabel("Discount factor D(t)", fontsize=11)
    ax.set_title(f"{label} — Valuation Fan Chart\n"
                 f"Uncertainty widens with horizon",
                 fontsize=12, fontweight="bold", pad=10)
    ax.legend(fontsize=9, framealpha=0.95, edgecolor="#dddddd",
              loc="upper right")
    ax.grid(True, alpha=0.3, linewidth=0.5)
    ax.spines[["top","right"]].set_visible(False)

fig.suptitle(
    "CIVI Valuation Fan Chart Under Parameter Uncertainty\n"
    "Bands show distributional spread of discount factor across N = 2,000 "
    "Monte Carlo draws  │  Wider band = greater valuation sensitivity",
    fontsize=12, fontweight="bold", y=1.03)

plt.tight_layout()
plt.savefig("fig7c_fan_chart.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure 7c saved.")

In [ ]:
# summary stats

rows = []
for arr, label in [(PV_civi_mc, "CIVI"), (PV_exp_mc, "Exponential")]:
    rows.append({
        "Method":    label,
        "Mean PV":   round(arr.mean(), 4),
        "Median PV": round(np.median(arr), 4),
        "Std Dev":   round(arr.std(), 4),
        "5th pct":   round(np.percentile(arr, 5), 4),
        "95th pct":  round(np.percentile(arr, 95), 4),
        "IQR":       round(np.percentile(arr,75) -
                           np.percentile(arr,25), 4),
        "CV":        round(arr.std() / arr.mean(), 4),
    })

df_mc = pd.DataFrame(rows).set_index("Method")
print("=" * 70)
print("MONTE CARLO SUMMARY  (DAC 300yr, N=10,000)")
print("=" * 70)
print(df_mc.to_string())
print()

cv_civi = PV_civi_mc.std() / PV_civi_mc.mean()
cv_exp  = PV_exp_mc.std()  / PV_exp_mc.mean()

print(f"  CIVI CV = {cv_civi:.4f}  |  Exp CV = {cv_exp:.4f}")
print(f"  CIVI is {'MORE' if cv_civi < cv_exp else 'LESS'} stable "
      f"than exponential under parameter uncertainty")
print()
print(f"  Mean PV ratio (CIVI / Exp) = "
      f"{PV_civi_mc.mean() / PV_exp_mc.mean():.4f}×")
print()

# Sensitivity Analysis

CIVI exposes two conceptually distinct layers to perturbation:

| Layer | Parameters | Method |
|---|---|---|
| **Empirical** | δ, η, g, p, L, k | Partial perturbation ±Δ around baseline |
| **Normative** | w₁…w₂₂ (moral weights) | +0.10 on each wᵢ, redistribute remainder |

These are analysed *separately* — conflating them would misrepresent the nature of normative disagreement as mere parameter uncertainty.


For each perturbation, the % change in PV relative to baseline is recorded:

$$\text{Sensitivity}_x = \frac{PV(x + \Delta x) - PV(x)}{PV(x)} \times 100$$

Results are combined into a tornado chart (ranked by absolute impact)
and a δ × η heatmap showing interaction effects between the two
most influential empirical parameters.

In [ ]:


X_dac = X_DAC    # 300yr benefit  stream

def build_civi_local(d, eta, g_, p_, L_, k_):
    """Build 6-function CIVI from given parameters."""
    r     = d + eta * g_
    r_lt  = max(d * 0.1, 0.001)
    r_sh  = min(r + 0.02, 0.12)
    r_lg  = max(r - 0.015, 0.001)

    D1 = np.exp(-r * t)
    D2 = np.exp(-(d * 0.1 + eta * g_) * t)
    D3 = 1.0 / (1.0 + k_ * t)
    D4 = np.exp(-(r + p_ * L_) * t)
    D5 = np.exp(-r_lt * t)
    D6 = np.where(
        t <= 30,
        np.exp(-r_sh * t),
        np.exp(-r_sh * 30) * np.exp(-r_lg * (t - 30))
    )
    return np.vstack([D1, D2, D3, D4, D5, D6]).mean(axis=0)

p0 = dict(d=delta_base, eta=eta_base, g_=g_base,
          p_=p_catastrophe, L_=L_loss, k_=k_hyperbolic)

CIVI_base = build_civi_local(**p0)
PV_base   = float(np.sum(CIVI_base * X_dac))

print(f"Baseline CIVI PV (DAC): {PV_base:.5f}")
print()

#  Perturbation grid
perturbations = {
    "δ (delta)":   ("d",   [-0.010, -0.005, +0.005, +0.010]),
    "η (eta)":     ("eta", [-0.500, -0.250, +0.250, +0.500]),
    "g (growth)":  ("g_",  [-0.010, -0.005, +0.005, +0.010]),
    "p (cat risk)":("p_",  [-0.005, -0.002, +0.002, +0.005]),
    "L (loss)":    ("L_",  [-0.100, -0.050, +0.050, +0.100]),
    "k (hyperb.)": ("k_",  [-0.010, -0.005, +0.005, +0.010]),
}

emp_results = {}   # label → {delta: pct_change}

print(f"  {'Parameter':<16} {'Perturbation':>14} "
      f"{'PV':>10}  {'% Change':>10}")
print("  " + "─" * 56)

for label, (key, deltas) in perturbations.items():
    emp_results[label] = {}
    for delta in deltas:
        p_pert = p0.copy()
        p_pert[key] = p0[key] + delta
        # Enforce bounds
        p_pert["d"]   = max(p_pert["d"],   0.0)
        p_pert["eta"] = np.clip(p_pert["eta"], 0.5, 4.0)
        p_pert["g_"]  = max(p_pert["g_"],  0.0)
        p_pert["p_"]  = max(p_pert["p_"],  0.0)
        p_pert["L_"]  = np.clip(p_pert["L_"], 0.01, 0.99)
        p_pert["k_"]  = max(p_pert["k_"],  0.001)

        civi_pert  = build_civi_local(**p_pert)
        pv_pert    = float(np.sum(civi_pert * X_dac))
        pct_change = (pv_pert - PV_base) / PV_base * 100
        emp_results[label][delta] = pct_change

        print(f"  {label:<16} {delta:>+14.4f} "
              f"{pv_pert:>10.4f}  {pct_change:>+10.3f}%")
    print()

#  Summary: max absolute impact per parameter
print("  Max absolute % impact per empirical parameter:")
emp_max = {}
for label, d in emp_results.items():
    max_abs = max(abs(v) for v in d.values())
    emp_max[label] = max_abs
    print(f"    {label:<18} {max_abs:.3f}%")

In [ ]:
# normative weight sensitivity

func_labels_local = [
    "w₁ Standard Exp.",
    "w₂ Near-Zero δ",
    "w₃ Hyperbolic",
    "w₄ Catastrophic",
    "w₅ Longtermist",
    "w₆ Dual-Rate",
]

DELTA_W   = 0.10      # weight perturbation
N_funcs   = 6
w_base    = np.ones(N_funcs) / N_funcs   # uniform baseline

weight_results = {}

print(f"Baseline weight per function: {1/N_funcs:.4f}")
print(f"Perturbation: +{DELTA_W} on one function, remainder renormalised")
print()
print(f"  {'Function':<26} {'w_base':>8} {'w_pert':>8} "
      f"{'PV_pert':>10} {'% Change':>10}")
print("  " + "─" * 66)

# Recompute local D array at baseline params
D1b = np.exp(-(delta_base + eta_base * g_base) * t)
D2b = np.exp(-(delta_base * 0.1 + eta_base * g_base) * t)
D3b = 1.0 / (1.0 + k_hyperbolic * t)
D4b = np.exp(-((delta_base + eta_base * g_base)
               + p_catastrophe * L_loss) * t)
D5b = np.exp(-max(delta_base * 0.1, 0.001) * t)
r_sh0 = min(delta_base + eta_base * g_base + 0.02, 0.12)
r_lg0 = max(delta_base + eta_base * g_base - 0.015, 0.001)
D6b = np.where(t <= 30,
               np.exp(-r_sh0 * t),
               np.exp(-r_sh0 * 30) * np.exp(-r_lg0 * (t - 30)))

D_local_base = np.vstack([D1b, D2b, D3b, D4b, D5b, D6b])  # (6, 301)

for i, label in enumerate(func_labels_local):
    # Build perturbed weight vector
    w_pert        = w_base.copy()
    w_pert[i]    += DELTA_W
    # Redistribute excess from remaining functions proportionally
    excess         = w_pert[i] - w_base[i]
    others         = [j for j in range(N_funcs) if j != i]
    for j in others:
        w_pert[j] = w_base[j] - excess / len(others)
    w_pert = np.clip(w_pert, 0, None)
    w_pert = w_pert / w_pert.sum()   # renormalise

    civi_pert  = np.dot(w_pert, D_local_base)
    pv_pert    = float(np.sum(civi_pert * X_dac))
    pct_change = (pv_pert - PV_base) / PV_base * 100

    weight_results[label] = {
        "w_base":     w_base[i],
        "w_pert":     w_pert[i],
        "pv_pert":    pv_pert,
        "pct_change": pct_change,
    }

    print(f"  {label:<26} {w_base[i]:>8.4f} {w_pert[i]:>8.4f} "
          f"{pv_pert:>10.4f} {pct_change:>+10.3f}%")

print()
print("  Max absolute % impact per weight perturbation:")
wt_max = {}
for label, d in weight_results.items():
    wt_max[label] = abs(d["pct_change"])
    print(f"    {label:<26}  {wt_max[label]:.3f}%")

In [ ]:
# tornado chart


fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.patch.set_facecolor("#fafafa")

EMP_COL  = "#2166ac"   # empirical parameters
NORM_COL = "#e05c39"   # normative weights

tornado_rows = []

for label, deltas_dict in emp_results.items():
    pos_vals = {d: v for d, v in deltas_dict.items() if d > 0}
    neg_vals = {d: v for d, v in deltas_dict.items() if d < 0}
    up   = max(pos_vals.values()) if pos_vals else 0
    down = min(neg_vals.values()) if neg_vals else 0
    tornado_rows.append({
        "label":    label,
        "up":       up,
        "down":     down,
        "max_abs":  max(abs(up), abs(down)),
        "layer":    "Empirical",
        "colour":   EMP_COL,
    })

# Normative: single perturbation (+0.10), so up = pct_change, down = ~−pct
for label, d in weight_results.items():
    pct = d["pct_change"]
    tornado_rows.append({
        "label":   label,
        "up":      max(pct, 0),
        "down":    min(pct, 0),
        "max_abs": abs(pct),
        "layer":   "Normative",
        "colour":  NORM_COL,
    })

# Sort by max absolute impact
tornado_rows.sort(key=lambda x: x["max_abs"])
labels_t = [r["label"]   for r in tornado_rows]
ups      = [r["up"]      for r in tornado_rows]
downs    = [r["down"]    for r in tornado_rows]
colours  = [r["colour"]  for r in tornado_rows]
layers   = [r["layer"]   for r in tornado_rows]

y_pos = np.arange(len(tornado_rows))

# tornado
ax = axes[0]
ax.set_facecolor("#f7f9fc")

for i, (label, up, down, col, layer) in enumerate(
        zip(labels_t, ups, downs, colours, layers)):

    # Positive bar
    if abs(up) > 0.001:
        ax.barh(i, up, color=col, alpha=0.80,
                edgecolor="white", linewidth=0.5, height=0.6)
        ax.text(up + 0.05, i, f"+{up:.2f}%",
                va="center", fontsize=8, color=col, fontweight="bold")

    # Negative bar
    if abs(down) > 0.001:
        ax.barh(i, down, color=col, alpha=0.45,
                edgecolor="white", linewidth=0.5, height=0.6)
        ax.text(down - 0.05, i, f"{down:.2f}%",
                va="center", fontsize=8, color=col,
                ha="right", fontweight="bold")

ax.axvline(0, color="#555555", linewidth=1.2, zorder=3)
ax.set_yticks(y_pos)
ax.set_yticklabels(labels_t, fontsize=9)
ax.set_xlabel("% change in CIVI Present Value", fontsize=11)
ax.set_title("Tornado Chart — Sensitivity of CIVI PV\n"
             "to Empirical and Normative Perturbations",
             fontsize=11, fontweight="bold", pad=10)

# Legend for layers
from matplotlib.patches import Patch
legend_els = [
    Patch(facecolor=EMP_COL,  alpha=0.80, label="Empirical parameter"),
    Patch(facecolor=NORM_COL, alpha=0.80, label="Normative weight (+0.10)"),
]
ax.legend(handles=legend_els, fontsize=9,
          framealpha=0.95, edgecolor="#dddddd", loc="lower right")
ax.grid(True, axis="x", alpha=0.3, linewidth=0.5)
ax.spines[["top","right"]].set_visible(False)

#  ranked bar of absolute impact
ax2 = axes[1]
ax2.set_facecolor("#f7f9fc")

# Sort descending for this panel
sorted_rows = sorted(tornado_rows, key=lambda x: x["max_abs"], reverse=True)
s_labels  = [r["label"]   for r in sorted_rows]
s_abs     = [r["max_abs"] for r in sorted_rows]
s_cols    = [r["colour"]  for r in sorted_rows]
s_layers  = [r["layer"]   for r in sorted_rows]

y2 = np.arange(len(sorted_rows))

bars = ax2.barh(y2, s_abs, color=s_cols, alpha=0.80,
                edgecolor="white", linewidth=0.5, height=0.6)

for bar, val, layer in zip(bars, s_abs, s_layers):
    ax2.text(val + 0.03, bar.get_y() + bar.get_height()/2,
             f"{val:.2f}%", va="center", fontsize=8.5,
             fontweight="bold", color="#222222")

ax2.set_yticks(y2)
ax2.set_yticklabels(s_labels, fontsize=9)
ax2.set_xlabel("Max absolute % impact on CIVI PV", fontsize=11)
ax2.set_title("Ranked Sensitivity — Absolute Impact\n"
              "Top = most influential parameter or weight",
              fontsize=11, fontweight="bold", pad=10)
ax2.legend(handles=legend_els, fontsize=9,
           framealpha=0.95, edgecolor="#dddddd", loc="lower right")
ax2.grid(True, axis="x", alpha=0.3, linewidth=0.5)
ax2.spines[["top","right"]].set_visible(False)

fig.suptitle("Tornado Chart: Sensitivity of CIVI Present Value "
             "to Parameter and Weight Perturbations",
             fontsize=13, fontweight="bold", y=1.02)

plt.tight_layout()
plt.savefig("fig8a_tornado.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure 9 saved.")

In [ ]:
# heatmap

#  Build grid
n_grid   = 15
d_grid   = np.linspace(0.001, 0.030, n_grid)
eta_grid = np.linspace(0.5,   4.0,   n_grid)

PV_surface = np.zeros((n_grid, n_grid))

for i, d_val in enumerate(d_grid):
    for j, eta_val in enumerate(eta_grid):
        civi_ij    = build_civi_local(
            d=d_val, eta=eta_val, g_=g_base,
            p_=p_catastrophe, L_=L_loss, k_=k_hyperbolic
        )
        PV_surface[i, j] = float(np.sum(civi_ij * X_dac))

#  Figure
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.patch.set_facecolor("#fafafa")

# PV heatmap
ax = axes[0]

im = ax.imshow(
    PV_surface,
    aspect="auto", origin="lower",
    extent=[eta_grid[0], eta_grid[-1], d_grid[0], d_grid[-1]],
    cmap="RdYlBu_r",
)

# Contour lines
CS = ax.contour(
    eta_grid, d_grid, PV_surface,
    levels=8, colors="white",
    linewidths=0.9, alpha=0.7,
)
ax.clabel(CS, inline=True, fontsize=7.5,
          fmt=lambda x: f"{x:.1f}")

# Mark baseline
ax.scatter([eta_base], [delta_base], color="white", s=120,
           zorder=5, marker="*",
           label=f"Baseline (δ={delta_base}, η={eta_base})")
ax.text(eta_base + 0.08, delta_base + 0.0008,
        "Baseline", fontsize=8.5, color="white",
        fontweight="bold")

cb = plt.colorbar(im, ax=ax, label="CIVI Present Value (DAC)",
                  shrink=0.88, pad=0.02)
cb.ax.tick_params(labelsize=8)

ax.set_xlabel("η — Elasticity of marginal utility", fontsize=11)
ax.set_ylabel("δ — Pure rate of time preference", fontsize=11)
ax.set_title("CIVI PV Surface across δ × η\n"
             "White star = baseline parameters",
             fontsize=11, fontweight="bold", pad=10)
ax.legend(fontsize=9, framealpha=0.90,
          edgecolor="#dddddd", loc="upper right")

# the % deviation from baseline
ax2 = axes[1]

PV_dev = (PV_surface - PV_base) / PV_base * 100

# Symmetric colour scale!
abs_max = np.abs(PV_dev).max()

im2 = ax2.imshow(
    PV_dev,
    aspect="auto", origin="lower",
    extent=[eta_grid[0], eta_grid[-1], d_grid[0], d_grid[-1]],
    cmap="RdBu_r",
    vmin=-abs_max, vmax=abs_max,
)

CS2 = ax2.contour(
    eta_grid, d_grid, PV_dev,
    levels=[-50, -30, -10, 0, 10, 30, 50],
    colors="white", linewidths=0.9, alpha=0.7,
)
ax2.clabel(CS2, inline=True, fontsize=7.5, fmt="%+.0f%%")

# Zero-deviation contour highlighted
ax2.contour(eta_grid, d_grid, PV_dev,
            levels=[0], colors=["#333333"],
            linewidths=2.0, linestyles="--")

ax2.scatter([eta_base], [delta_base], color="#333333",
            s=120, zorder=5, marker="*")
ax2.text(eta_base + 0.08, delta_base + 0.0008,
         "Baseline", fontsize=8.5, color="#333333",
         fontweight="bold")

cb2 = plt.colorbar(im2, ax=ax2, label="% deviation from baseline PV",
                   shrink=0.88, pad=0.02)
cb2.ax.tick_params(labelsize=8)

ax2.set_xlabel("η — Elasticity of marginal utility", fontsize=11)
ax2.set_ylabel("δ — Pure rate of time preference", fontsize=11)
ax2.set_title("% Deviation from Baseline PV\n"
              "Dashed line = zero deviation  │  Red = lower PV",
              fontsize=11, fontweight="bold", pad=10)

fig.suptitle(
    "CIVI Present Value Surface across δ and η Parameter Space\n"
    "Left: absolute PV  │  Right: % deviation from baseline",
    fontsize=12, fontweight="bold", y=1.02)

plt.tight_layout()
plt.savefig("fig8b_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure 8b saved.")

In [ ]:
# weight elasticity

w_sweep    = np.linspace(0, 1, 61)
fn_colours = ["#e05c39", "#d4a017", "#2a7a3b",
              "#c0392b", "#2166ac", "#9b59b6"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
fig.patch.set_facecolor("#fafafa")

pv_curves = {}

#  LEFT: PV vs weight for each function
ax = axes[0]
ax.set_facecolor("#f7f9fc")

for i, (label, col) in enumerate(
        zip(func_labels_local, fn_colours)):

    pv_vals = []
    for wi in w_sweep:
        w_tmp     = np.ones(N_funcs) * (1 - wi) / max(N_funcs - 1, 1)
        w_tmp[i]  = wi
        w_tmp     = np.clip(w_tmp, 0, None)
        w_tmp    /= w_tmp.sum()
        civi_tmp  = np.dot(w_tmp, D_local_base)
        pv_vals.append(float(np.sum(civi_tmp * X_dac)))

    pv_vals = np.array(pv_vals)
    pv_curves[label] = pv_vals

    ax.plot(w_sweep, pv_vals, color=col, linewidth=2.2,
            label=label)

# Mark uniform weight
ax.axvline(1/N_funcs, color="#888888", linewidth=1.3,
           linestyle=":", label=f"Uniform (1/{N_funcs})")

ax.set_xlim(0, 1)
ax.set_xlabel(f"Weight allocated to function", fontsize=11)
ax.set_ylabel("CIVI Present Value (DAC, 300yr)", fontsize=11)
ax.set_title("PV vs Moral Weight Allocation\n"
             "(remainder split uniformly across other 5)",
             fontsize=11, fontweight="bold", pad=10)
ax.legend(fontsize=8.5, framealpha=0.95,
          edgecolor="#dddddd", loc="upper left")
ax.grid(True, alpha=0.3, linewidth=0.5)
ax.spines[["top","right"]].set_visible(False)

#  RIGHT: elasticity (bascially dPV/dw normalised)
ax2 = axes[1]
ax2.set_facecolor("#f7f9fc")

unif_idx = np.argmin(np.abs(w_sweep - 1/N_funcs))

for i, (label, col) in enumerate(
        zip(func_labels_local, fn_colours)):

    pv_vals    = pv_curves[label]
    elasticity = np.gradient(pv_vals, w_sweep)

    ax2.plot(w_sweep, elasticity, color=col,
             linewidth=2.2, label=label)

ax2.axvline(1/N_funcs, color="#888888", linewidth=1.3,
            linestyle=":", label=f"Uniform weight")
ax2.axhline(0, color="#555555", linewidth=1.0,
            linestyle="--", alpha=0.6)

ax2.set_xlim(0, 1)
ax2.set_xlabel("Weight allocated to function", fontsize=11)
ax2.set_ylabel("dPV / dw  (marginal PV per unit weight)", fontsize=11)
ax2.set_title("Marginal PV Elasticity w.r.t. Moral Weight\n"
              "High elasticity = high normative leverage",
              fontsize=11, fontweight="bold", pad=10)
ax2.legend(fontsize=8.5, framealpha=0.95,
           edgecolor="#dddddd", loc="upper right")
ax2.grid(True, alpha=0.3, linewidth=0.5)
ax2.spines[["top","right"]].set_visible(False)

fig.suptitle(
    "Figure 8c — Moral Weight Elasticity of CIVI Present Value\n"
    "Left: PV trajectory as weight shifts  │  "
    "Right: marginal impact of additional credence",
    fontsize=12, fontweight="bold", y=1.02)

plt.tight_layout()
plt.savefig("fig8c_weight_elasticity.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure 8c saved.")

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 8.6 │ Section 8 Summary
# ─────────────────────────────────────────────────────────────

print("=" * 68)
print("  Key findings from this sensitivity analysis")
print("=" * 68)

print(f"\n  Baseline CIVI PV (DAC, 300yr): {PV_base:.4f}")
print(f"  Exponential PV (baseline):     "
      f"{float(np.sum(D_exponential * X_dac)):.4f}")

print("\n  EMPIRICAL LAYER — % impact of ±perturbation:")
print(f"  {'Parameter':<18} {'Max |% change|':>16}  {'Rank'}")
print("  " + "─" * 44)
emp_sorted = sorted(emp_max.items(), key=lambda x: x[1], reverse=True)
for rank, (label, val) in enumerate(emp_sorted, 1):
    print(f"  {label:<18} {val:>14.3f}%  #{rank}")

print("\n  NORMATIVE LAYER — % impact of +0.10 weight shift:")
print(f"  {'Function':<28} {'|% change|':>12}  {'Rank'}")
print("  " + "─" * 48)
wt_sorted = sorted(wt_max.items(), key=lambda x: x[1], reverse=True)
for rank, (label, val) in enumerate(wt_sorted, 1):
    print(f"  {label:<28} {val:>10.3f}%  #{rank}")

# Catastrophic Tail Scenario

The standard Monte Carlo done propagates parameter uncertainty
while keeping the benefit stream $X(t)$ intact.

This section models
a qualitatively different risk: a **catastrophic shock that disrupts the benefit stream itself** at a random point in the horizon.

A catastrophe occurs at random time $\tau$:

$$\tau \sim \text{Uniform}(1, 300)$$

The welfare loss on catastrophe is fat-tailed:

$$L \sim \text{LogNormal}(\mu = \log(0.30),\; \sigma = 0.80)$$

The shocked benefit stream becomes:

$$\tilde{X}(t) = \begin{cases} X(t) & t < \tau \\ X(t) \cdot (1 - L) & t \geq \tau \end{cases}$$

In the **hard shock** variant, $L = 1$ — the stream terminates entirely at $\tau$.

Does CIVI's pluralist structure provide resilience under tail shocks,
or does its higher weighting of the far future make it *more* exposed
than exponential discounting when long-run benefit streams collapse?

In [ ]:
from scipy.stats import lognorm as lognorm_dist
from tqdm.auto import tqdm

np.random.seed(123)

N_tail = 10_000
X_dac  = X_DAC    # 300yr benefit stream

results_tail = {
    "soft": {
        "PV_civi": np.zeros(N_tail),
        "PV_exp":  np.zeros(N_tail),
        "tau":     np.zeros(N_tail),
        "L":       np.zeros(N_tail),
    },
    "hard": {
        "PV_civi": np.zeros(N_tail),
        "PV_exp":  np.zeros(N_tail),
        "tau":     np.zeros(N_tail),
    },
}

tau_draws = np.random.randint(1, 301, size=N_tail)   # uniform over [1,300]
L_draws   = np.clip(
    lognorm_dist.rvs(s=0.80, scale=0.30, size=N_tail),
    0.05, 0.99
)

print(f"Catastrophe timing:  mean τ = {tau_draws.mean():.1f}  "
      f"(uniform [1, 300])")
print(f"Welfare loss (soft): mean L = {L_draws.mean():.3f}  "
      f"| 95th pct = {np.percentile(L_draws, 95):.3f}")
print(f"                     P(L > 0.8) = "
      f"{(L_draws > 0.8).mean():.3f}  ← fat tail check")
print()
print("Running tail simulations...")

for sim in tqdm(range(N_tail)):

    tau = tau_draws[sim]
    L   = L_draws[sim]

    d   = float(sample_delta(1))
    eta = float(sample_eta(1))
    g_  = float(sample_g(1))
    p_  = float(sample_p_cat(1))
    Lp  = float(sample_L(1))
    k_  = float(sample_k(1))

    civi_d = build_civi_local(d=d, eta=eta, g_=g_,
                               p_=p_, L_=Lp, k_=k_)
    exp_d  = np.exp(-(d + eta * g_) * t)

    #  Soft shock stream
    X_soft         = X_dac.copy().astype(float)
    X_soft[tau:]  *= (1.0 - L)

    results_tail["soft"]["PV_civi"][sim] = float(
        np.sum(civi_d * X_soft))
    results_tail["soft"]["PV_exp"][sim]  = float(
        np.sum(exp_d  * X_soft))
    results_tail["soft"]["tau"][sim]     = tau
    results_tail["soft"]["L"][sim]       = L

    #  Hard shock stream (termination)
    X_hard        = X_dac.copy().astype(float)
    X_hard[tau:] *= 0.0

    results_tail["hard"]["PV_civi"][sim] = float(
        np.sum(civi_d * X_hard))
    results_tail["hard"]["PV_exp"][sim]  = float(
        np.sum(exp_d  * X_hard))
    results_tail["hard"]["tau"][sim]     = tau

print("\nSimulation complete.")
print()

# summary
print("=" * 68)
print("  TAIL SCENARIO SUMMARY  (N = 10,000, DAC 300yr)")
print("=" * 68)
print()

for scenario, label in [("soft", "Soft shock (stream scaled)"),
                         ("hard", "Hard shock (termination)")]:
    r = results_tail[scenario]
    print(f"  {label}")
    print(f"  {'':4} {'':10} {'CIVI':>12} {'Exponential':>14}")
    print("  " + "─" * 42)
    for stat, fn in [
        ("Mean",   np.mean),
        ("Median", np.median),
        ("Std",    np.std),
        ("5th",    lambda x: np.percentile(x, 5)),
        ("95th",   lambda x: np.percentile(x, 95)),
        ("CV",     lambda x: np.std(x)/np.mean(x)),
    ]:
        vc = fn(r["PV_civi"])
        ve = fn(r["PV_exp"])
        print(f"  {'':4} {stat:<10} {vc:>12.4f} {ve:>14.4f}")
    print()

# Baseline reference
print(f"  Baseline (no shock) CIVI PV: {PV_base:.4f}")
print(f"  Baseline (no shock) Exp  PV: "
      f"{float(np.sum(D_exponential * X_dac)):.4f}")

In [ ]:
# simplified violin plot

import matplotlib.patches as mpatches

fig, axes = plt.subplots(1, 2, figsize=(15, 6.5))
fig.patch.set_facecolor("#fafafa")

BLUE  = "#2166ac"
RED   = "#d62728"
LBLUE = "#9ecae1"
LRED  = "#fc9272"

scenario_labels = ["Baseline\n(no shock)",
                   "Soft shock\n(partial loss)",
                   "Hard shock\n(termination)"]

for ax, (method, col, lcol) in zip(
    axes,
    [("civi", BLUE, LBLUE), ("exp", RED, LRED)]
):
    ax.set_facecolor("#f7f9fc")

    data_sets = [
        PV_civi_mc if method == "civi" else PV_exp_mc,
        results_tail["soft"][f"PV_{method}"],
        results_tail["hard"][f"PV_{method}"],
    ]

    positions = [1, 2, 3]

    parts = ax.violinplot(
        data_sets,
        positions=positions,
        widths=0.55,
        showmedians=True,
        showextrema=False,
    )

    alphas = [0.85, 0.65, 0.45]
    for i, (pc, alpha) in enumerate(zip(parts["bodies"], alphas)):
        pc.set_facecolor(col)
        pc.set_alpha(alpha)
        pc.set_edgecolor("white")
        pc.set_linewidth(0.5)

    parts["cmedians"].set_color("white")
    parts["cmedians"].set_linewidth(2.5)

    # IQR boxes overlay
    for i, (data, pos) in enumerate(zip(data_sets, positions)):
        q25, q50, q75 = np.percentile(data, [25, 50, 75])
        p5,  p95       = np.percentile(data, [5, 95])

        # Box
        ax.add_patch(mpatches.FancyBboxPatch(
            (pos - 0.12, q25), 0.24, q75 - q25,
            boxstyle="round,pad=0.01",
            facecolor="white", edgecolor=col,
            linewidth=1.5, alpha=0.6, zorder=3
        ))

        # 90% whiskers
        ax.plot([pos, pos], [p5, q25],
               color=col, linewidth=1.5, alpha=0.7)
        ax.plot([pos, pos], [q75, p95],
                color=col, linewidth=1.5, alpha=0.7)
        ax.plot([pos - 0.1, pos + 0.1], [p5,  p5],
               color=col, linewidth=1.5, alpha=0.7)
        ax.plot([pos - 0.1, pos + 0.1], [p95, p95],
                color=col, linewidth=1.5, alpha=0.7)


        # Mean dot
        ax.scatter([pos], [data.mean()], color=col,
                   s=50, zorder=5, edgecolors="white",
                   linewidths=1.2)

        # % drop annotation relative to baseline
        if i > 0:
            base_mean = (PV_civi_mc if method == "civi"
                         else PV_exp_mc).mean()
            drop = (data.mean() - base_mean) / base_mean * 100
            ax.text(pos, ax.get_ylim()[1] * 0.97
                    if ax.get_ylim()[1] > 0 else data.max() * 1.05,
                    f"{drop:+.1f}%",
                    ha="center", fontsize=9, color=col,
                    fontweight="bold")

    ax.set_xticks(positions)
    ax.set_xticklabels(scenario_labels, fontsize=10)
    ax.set_ylabel("Present Value (discounted tonne-years)", fontsize=11)
    label_str = "CIVI" if method == "civi" else "Exponential"
    ax.set_title(f"{label_str} — PV Distribution\nAcross Shock Scenarios",
                 fontsize=12, fontweight="bold", pad=10)
    ax.grid(True, axis="y", alpha=0.3, linewidth=0.5)
    ax.spines[["top", "right"]].set_visible(False)

fig.suptitle(
    "Present Value Distributions: "
    "Baseline vs Catastrophic Tail Scenarios\n"
    "Width of violin = density  │  Box = IQR  │  "
    "Dot = mean  │  % = change from baseline mean",
    fontsize=12, fontweight="bold", y=1.03)

plt.tight_layout()
plt.savefig("fig9a_tail_violin.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure 9a saved.")

In [ ]:
# shock timing analysis

tau_bins  = np.arange(0, 301, 30)
bin_mids  = (tau_bins[:-1] + tau_bins[1:]) / 2

mean_pv_civi_by_tau = []
mean_pv_exp_by_tau  = []

tau_vals_s  = results_tail["hard"]["tau"]
pv_civi_s   = results_tail["hard"]["PV_civi"]
pv_exp_s    = results_tail["hard"]["PV_exp"]

for lo, hi in zip(tau_bins[:-1], tau_bins[1:]):
    mask = (tau_vals_s >= lo) & (tau_vals_s < hi)
    if mask.sum() > 0:
        mean_pv_civi_by_tau.append(pv_civi_s[mask].mean())
        mean_pv_exp_by_tau.append(pv_exp_s[mask].mean())
    else:
        mean_pv_civi_by_tau.append(np.nan)
        mean_pv_exp_by_tau.append(np.nan)

mean_pv_civi_by_tau = np.array(mean_pv_civi_by_tau)
mean_pv_exp_by_tau  = np.array(mean_pv_exp_by_tau)

#  Also compute deterministic PV(τ) — stream cut at each τ
tau_range      = np.arange(1, 301)
det_pv_civi    = np.array([
    float(np.sum(CIVI_local[:tau+1] * X_dac[:tau+1]))
    for tau in tau_range
])
det_pv_exp     = np.array([
    float(np.sum(D_exponential[:tau+1] * X_dac[:tau+1]))
    for tau in tau_range
])

# Normalise to fraction of full PV (no shock)
full_pv_civi = float(np.sum(CIVI_local * X_dac))
full_pv_exp  = float(np.sum(D_exponential * X_dac))

frac_civi = det_pv_civi / full_pv_civi
frac_exp  = det_pv_exp  / full_pv_exp

#  Figure
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.patch.set_facecolor("#fafafa")

BLUE = "#2166ac"
RED  = "#d62728"
GREY = "#888888"

ax = axes[0]
ax.set_facecolor("#f7f9fc")

ax.fill_between(tau_range, frac_civi, frac_exp,
                where=(frac_civi >= frac_exp),
                alpha=0.15, color=BLUE, interpolate=True)
ax.fill_between(tau_range, frac_civi, frac_exp,
                where=(frac_civi < frac_exp),
                alpha=0.15, color=RED, interpolate=True)

ax.plot(tau_range, frac_exp,  color=RED,  linewidth=2.4,
        linestyle="--", label="Exponential")
ax.plot(tau_range, frac_civi, color=BLUE, linewidth=2.6,
        label="CIVI")

# Mark 50% recovery point
for arr, col, method in [
    (frac_civi, BLUE, "CIVI"),
    (frac_exp,  RED,  "Exp")
]:
    idx_50 = np.argmin(np.abs(arr - 0.50))
    ax.scatter([tau_range[idx_50]], [0.50], color=col,
               s=80, zorder=5)
    ax.annotate(f"{method}: 50%\nat t={tau_range[idx_50]}",
                xy=(tau_range[idx_50], 0.50),
                xytext=(tau_range[idx_50] + 20,
                        0.50 + (0.06 if method == "CIVI" else -0.08)),
                fontsize=8.5, color=col, fontweight="bold",
                arrowprops=dict(arrowstyle="->",
                                color=col, lw=1.2))

ax.axhline(0.50, color=GREY, linewidth=1.0,
           linestyle=":", alpha=0.7, label="50% recovery threshold")

ax.set_xlim(0, 300)
ax.set_ylim(0, 1.05)
ax.set_xlabel("Catastrophe year τ", fontsize=11)
ax.set_ylabel("PV recovered as fraction of no-shock PV", fontsize=11)
ax.set_title("Fraction of Full PV Recovered\nif Hard Shock Hits at Year τ",
             fontsize=11, fontweight="bold", pad=10)
ax.legend(fontsize=9.5, framealpha=0.95, edgecolor="#dddddd")
ax.grid(True, alpha=0.3, linewidth=0.5)
ax.spines[["top", "right"]].set_visible(False)

ax2 = axes[1]
ax2.set_facecolor("#f7f9fc")

# Marginal PV gained by surviving one more year
marginal_civi = np.diff(det_pv_civi)   # PV(τ) - PV(τ-1)
marginal_exp  = np.diff(det_pv_exp)

ax2.fill_between(tau_range[1:], marginal_civi, marginal_exp,
                 where=(marginal_civi >= marginal_exp),
                 alpha=0.18, color=BLUE, interpolate=True)
ax2.fill_between(tau_range[1:], marginal_civi, marginal_exp,
                 where=(marginal_civi < marginal_exp),
                 alpha=0.18, color=RED, interpolate=True)

ax2.plot(tau_range[1:], marginal_exp,  color=RED,  linewidth=2.2,
         linestyle="--", label="Exponential — marginal year value")
ax2.plot(tau_range[1:], marginal_civi, color=BLUE, linewidth=2.4,
         label="CIVI — marginal year value")

# Annotate where marginal CIVI > marginal Exp
cross = np.where(marginal_civi > marginal_exp)[0]
if len(cross) > 0:
    yr = tau_range[1:][cross[0]]
    ax2.axvline(yr, color=GREY, linewidth=1.2,
                linestyle=":", alpha=0.8)
    ax2.text(yr + 5, marginal_civi[cross[0]],
             f"CIVI > Exp\nfrom t={yr}",
             fontsize=8.5, color="#333333")

ax2.set_xlim(0, 300)
ax2.set_xlabel("Storage year τ", fontsize=11)
ax2.set_ylabel("Marginal PV of surviving one additional year",
               fontsize=11)
ax2.set_title("Marginal Value of One More Year of Storage\n"
              "CIVI preserves marginal value at long horizons",
              fontsize=11, fontweight="bold", pad=10)
ax2.legend(fontsize=9.5, framealpha=0.95, edgecolor="#dddddd")
ax2.grid(True, alpha=0.3, linewidth=0.5)
ax2.spines[["top", "right"]].set_visible(False)

fig.suptitle(
    "Shock Timing Analysis: "
    "How Catastrophe Year τ Determines PV Loss\n"
    "Exponential pre-discounts the far future; "
    "CIVI remains genuinely exposed to late shocks",
    fontsize=12, fontweight="bold", y=1.03)

plt.tight_layout()
plt.savefig("fig9b_shock_timing.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure 9b saved.")

In [ ]:
# joint distribution map
# This is the honest cost of caring about the long run!


# Compute PV loss % relative to no-shock baseline
tau_s      = results_tail["soft"]["tau"]
pv_civi_soft = results_tail["soft"]["PV_civi"]
pv_exp_soft  = results_tail["soft"]["PV_exp"]
L_s        = results_tail["soft"]["L"]

loss_civi  = (full_pv_civi - pv_civi_soft) / full_pv_civi * 100
loss_exp   = (full_pv_exp  - pv_exp_soft)  / full_pv_exp  * 100

fig = plt.figure(figsize=(16, 7))
fig.patch.set_facecolor("#fafafa")

gs = fig.add_gridspec(2, 3,
                      width_ratios=[1, 1, 0.55],
                      height_ratios=[1, 1],
                      hspace=0.08, wspace=0.32)

ax_civi  = fig.add_subplot(gs[:, 0])   # CIVI hexbin (full height)
ax_exp   = fig.add_subplot(gs[:, 1])   # Exp hexbin  (full height)
ax_top   = fig.add_subplot(gs[0, 2])   # τ marginal
ax_bot   = fig.add_subplot(gs[1, 2])   # loss marginal

BLUE = "#2166ac"
RED  = "#d62728"

#  CIVI hexbin
hb_civi = ax_civi.hexbin(
    tau_s, loss_civi,
    gridsize=35, cmap="Blues",
    mincnt=1, linewidths=0.2,
)
ax_civi.set_xlabel("Catastrophe year τ", fontsize=11)
ax_civi.set_ylabel("PV loss (%)", fontsize=11)
ax_civi.set_title("CIVI\nJoint distribution of (τ, PV loss %)",
                  fontsize=11, fontweight="bold", pad=10)
ax_civi.set_facecolor("#f0f4ff")

# Median loss by τ quintile
for q_lo, q_hi in zip([0,60,120,180,240], [60,120,180,240,300]):
    m = (tau_s >= q_lo) & (tau_s < q_hi)
    if m.sum() > 0:
        ax_civi.scatter(
            [(q_lo + q_hi) / 2], [np.median(loss_civi[m])],
            color="white", s=80, edgecolors=BLUE,
            linewidth=2.0, zorder=5
        )

plt.colorbar(hb_civi, ax=ax_civi, label="Count", shrink=0.7, pad=0.02)
ax_civi.spines[["top", "right"]].set_visible(False)

#  Exponential hexbin
hb_exp = ax_exp.hexbin(
    tau_s, loss_exp,
    gridsize=35, cmap="Reds",
    mincnt=1, linewidths=0.2,
)
ax_exp.set_xlabel("Catastrophe year τ", fontsize=11)
ax_exp.set_ylabel("PV loss (%)", fontsize=11)
ax_exp.set_title("Exponential\nJoint distribution of (τ, PV loss %)",
                 fontsize=11, fontweight="bold", pad=10)
ax_exp.set_facecolor("#fff0f0")

for q_lo, q_hi in zip([0,60,120,180,240], [60,120,180,240,300]):
    m = (tau_s >= q_lo) & (tau_s < q_hi)
    if m.sum() > 0:
        ax_exp.scatter(
            [(q_lo + q_hi) / 2], [np.median(loss_exp[m])],
            color="white", s=80, edgecolors=RED,
            linewidth=2.0, zorder=5
        )

plt.colorbar(hb_exp, ax=ax_exp, label="Count", shrink=0.7, pad=0.02)
ax_exp.spines[["top", "right"]].set_visible(False)

ax_top.set_facecolor("#f7f9fc")
ax_top.hist(tau_s, bins=40, color=GREY,
            alpha=0.70, edgecolor="white", linewidth=0.3)
ax_top.set_xlabel("Catastrophe year τ", fontsize=9.5)
ax_top.set_ylabel("Count", fontsize=9.5)
ax_top.set_title("τ distribution\n(Uniform[1,300])",
                 fontsize=9.5, fontweight="bold")
ax_top.grid(True, alpha=0.3, linewidth=0.5)
ax_top.spines[["top", "right"]].set_visible(False)

ax_bot.set_facecolor("#f7f9fc")

bins_loss = np.linspace(0, max(loss_civi.max(), loss_exp.max()), 60)
ax_bot.hist(loss_exp,  bins=bins_loss, color=RED,  alpha=0.50,
            label="Exponential", edgecolor="white", linewidth=0.3)
ax_bot.hist(loss_civi, bins=bins_loss, color=BLUE, alpha=0.60,
            label="CIVI", edgecolor="white", linewidth=0.3)

ax_bot.axvline(loss_civi.mean(), color=BLUE, linewidth=2.0,
               linestyle="--")
ax_bot.axvline(loss_exp.mean(),  color=RED,  linewidth=2.0,
               linestyle="--")
ax_bot.set_xlabel("PV loss (%)", fontsize=9.5)
ax_bot.set_ylabel("Count", fontsize=9.5)
ax_bot.set_title("PV loss distribution\nCIVI vs Exponential",
                 fontsize=9.5, fontweight="bold")
ax_bot.legend(fontsize=8, framealpha=0.9)
ax_bot.grid(True, alpha=0.3, linewidth=0.5)
ax_bot.spines[["top", "right"]].set_visible(False)

fig.suptitle(
    "Joint Distribution of Catastrophe Timing and PV Loss\n"
    "White circles = median PV loss per τ-quintile  │  "
    "CIVI loss is spread across the horizon; "
    "Exponential loss front-loaded",
    fontsize=12, fontweight="bold", y=1.02)

plt.savefig("fig9c_joint_distribution.png", dpi=150,
            bbox_inches="tight")
plt.show()
print("Figure 9c saved.")

In [ ]:
# key findings of fat tail risks

pv_base_exp = float(np.sum(D_exponential * X_dac))

for scenario, label in [
    ("soft", "Soft shock (partial, fat-tailed L)"),
    ("hard", "Hard shock (stream termination)"),
]:
    r = results_tail[scenario]
    drop_civi = (r["PV_civi"].mean() - full_pv_civi) / full_pv_civi * 100
    drop_exp  = (r["PV_exp"].mean()  - pv_base_exp)  / pv_base_exp  * 100

    print(f"\n  {label}")
    print(f"  {'':4} {'':20} {'CIVI':>10} {'Exp':>10}")
    print("  " + "─" * 46)
    print(f"  {'':4} {'Baseline (no shock)':<20} "
          f"{full_pv_civi:>10.3f} {pv_base_exp:>10.3f}")
    print(f"  {'':4} {'Mean PV under shock':<20} "
          f"{r['PV_civi'].mean():>10.3f} "
          f"{r['PV_exp'].mean():>10.3f}")
    print(f"  {'':4} {'Mean PV drop':<20} "
          f"{drop_civi:>+10.1f}% {drop_exp:>+10.1f}%")
    print(f"  {'':4} {'CV under shock':<20} "
          f"{r['PV_civi'].std()/r['PV_civi'].mean():>10.4f} "
          f"{r['PV_exp'].std()/r['PV_exp'].mean():>10.4f}")

In [1]:
from google.colab import _message
import copy

# Get full notebook JSON
nb = _message.blocking_request("get_ipynb")["ipynb"]

# Create a completely fresh notebook structure
clean_nb = {
    "cells": [],
    "metadata": {},   # <-- no widgets metadata at all
    "nbformat": 4,
    "nbformat_minor": 5
}

# Copy cells WITHOUT widget metadata
for cell in nb["cells"]:
    new_cell = copy.deepcopy(cell)
    new_cell["metadata"] = {}  # strip all cell-level metadata
    clean_nb["cells"].append(new_cell)

# Replace notebook in frontend
_message.blocking_request("set_ipynb", {"ipynb": clean_nb})

print("Notebook fully rebuilt without widget metadata.")

Notebook fully rebuilt without widget metadata.
